In [0]:
"""Post-refresh work that genuinely remains outside the Silver Journey DLT graph.

Run this once after a successful Silver Journey pipeline update. The DLT graph
already computes document corpus frequency and latest document versions, so this
notebook does not build reference tables or trigger a second pipeline update.
It publishes the reviewed form-instrument tables and verifies/repairs the
physical patient-event clustering.
"""

In [0]:
import json
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# This notebook is the production post-refresh artefact, so its concrete
# source/output tables are deliberate. There are no widgets or job parameters.
SOURCE_TABLE = "4_prod.silver.clinical_form"
MAP_LANE = "4_prod.bronze.map_powerform_assessment_item"
TARGET_SCHEMA = "4_prod.silver"
STAGING_SCHEMA = "4_prod.tmp"

In [0]:
PROVENANCE_COLS = [
    "QUESTION_CONCEPT_ID",
    "VALUE_CONCEPT_ID",
    "UNIT_CONCEPT_ID",
    "QUESTION_MAPPING_SOURCE",
    "VALUE_MAPPING_SOURCE",
    "POWERFORM_MAPPING_RULE_ID",
    "POWERFORM_MAPPING_VERSION",
    "CANONICAL_VALUE_MAPPING_RULE_ID",
    "CANONICAL_MATCH_STATUS",
]

RESPONSE_SCHEMA = T.ArrayType(
    T.StructType(
        [
            T.StructField("sequence", T.LongType()),
            T.StructField("response_id", T.StringType()),
            T.StructField("section", T.StringType()),
            T.StructField("section_id", T.StringType()),
            T.StructField("element_id", T.StringType()),
            T.StructField("element_label", T.StringType()),
            T.StructField("grid_name", T.StringType()),
            T.StructField("grid_row", T.StringType()),
            T.StructField("response_kind", T.StringType()),
            T.StructField("response_value_text", T.StringType()),
            T.StructField("response_value_number", T.DecimalType(38, 10)),
            T.StructField("response_value_datetime", T.TimestampType()),
            T.StructField("response_coding_system", T.StringType()),
            T.StructField("response_coding_code", T.StringType()),
            T.StructField("response_coding_display", T.StringType()),
            T.StructField("question_concept_id", T.StringType()),
            T.StructField("value_concept_id", T.StringType()),
            T.StructField("unit_concept_id", T.StringType()),
            T.StructField("pregnancy_id", T.StringType()),
            T.StructField("pregnancy_match_method", T.StringType()),
            T.StructField("active_ind", T.BooleanType()),
            T.StructField("source_present_ind", T.BooleanType()),
            T.StructField("source_deleted_ind", T.BooleanType()),
            T.StructField("canonical_match_status", T.StringType()),
        ]
    )
)

HOUSE_COLUMNS = [
    "patient_event_id",
    "fact_row_id",
    "subject_key",
    "subject_id_system",
    "person_id",
    "identity_status",
    "encounter_id",
    "event_datetime",
    "event_end_datetime",
    "authored_datetime",
    "completed_datetime",
    "performed_practitioner_id",
    "organization_id",
    "source_coding_system",
    "source_code",
    "source_display",
    "form_type_code",
    "form_type_display",
    "form_status_code",
    "form_status_display",
    "record_status",
    "record_status_effective_from",
    "record_status_effective_to",
    "response_row_count",
    "active_response_row_count",
    "empty_response_row_count",
    "invalid_response_row_count",
    "matched_response_row_count",
    "unmatched_response_row_count",
    "context_conflict_ind",
    "context_quarantined_ind",
    "confidentiality_code",
    "vip_ind",
    "withheld_identity_ind",
    "source_feed",
    "load_batch_id",
    "source_update_timestamp",
    "loaded_at",
]

In [0]:
# Reviewed form definitions, expanded for inspection and editing.

FORM_CONFIGS = []

In [0]:
FORM_CONFIGS.append(
{'slug': 'audit_c',
 'display': 'AUDIT-C alcohol screening (both form eras)',
 'source_displays': ['Alcohol Screening and Referral Form - AUDIT-C', 'Audit C Form'],
 'elements': [{'label': 'AuditC Alcoholic Drink Frequency',
               'matches': [{'label': 'AuditC Alcoholic Drink Frequency',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Alcoholic Drink Frequency',
                            'section': 'Audit C - Basic'}],
               'column': 'drink_frequency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Units Per Drink Day',
               'matches': [{'label': 'AuditC Units Per Drink Day',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Units Per Drink Day', 'section': 'Audit C - Basic'}],
               'column': 'units_per_drink_day',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC 6 Or 8 More Units On An Occaision',
               'matches': [{'label': 'AuditC 6 Or 8 More Units On An Occaision',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC 6 Or 8 More Units On An Occaision',
                            'section': 'Audit C - Basic'}],
               'column': 'six_plus_units_frequency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Basic Score',
               'matches': [{'label': 'AuditC Basic Score',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Basic Score', 'section': 'Audit C - Basic'}],
               'column': 'basic_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'AuditC Scoring Guidance Calc',
               'matches': [{'label': 'AuditC Scoring Guidance Calc',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Scoring Guidance Calc', 'section': 'Audit C - Basic'}],
               'column': 'scoring_guidance_calc',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'AuditC Basic 5-10',
               'matches': [{'label': 'AuditC Basic 5-10',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Basic 5-10', 'section': 'Audit C - Basic'}],
               'column': 'actions_score_5_10',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'AuditC Basic 11-12',
               'matches': [{'label': 'AuditC Basic 11-12',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Basic 11-12', 'section': 'Audit C - Basic'}],
               'column': 'actions_score_11_12',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'AuditC specialist referral',
               'matches': [{'label': 'AuditC specialist referral',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC specialist referral', 'section': 'Audit C - Basic'}],
               'column': 'specialist_referral',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'AuditC Referral Site Cal',
               'matches': [{'label': 'AuditC Referral Site Cal',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Referral Site Cal', 'section': 'Audit C - Basic'}],
               'column': 'referral_site_calc',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'AuditC Contact Number',
               'matches': [{'label': 'AuditC Contact Number',
                            'section': 'Alcohol Screening and Referral Form - AUDIT-C'},
                           {'label': 'AuditC Contact Number', 'section': 'Audit C - Basic'},
                           {'label': 'AuditC Contact Number', 'section': 'Audit C - Positive'}],
               'column': 'contact_number',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'AuditC Consent to ALN',
               'section': 'Audit C - Basic',
               'column': 'consent_to_aln',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Not Able to Stop Drinking',
               'section': 'Audit C - Positive',
               'column': 'unable_to_stop_drinking',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Failed to do Because Drinking',
               'section': 'Audit C - Positive',
               'column': 'failed_duties_due_to_drinking',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Frequency Needed Drink in Morning',
               'section': 'Audit C - Positive',
               'column': 'morning_drink_frequency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Frequency Guilt Afer Drinking',
               'section': 'Audit C - Positive',
               'column': 'guilt_after_drinking_frequency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Frequency Unable to Remember',
               'section': 'Audit C - Positive',
               'column': 'memory_loss_frequency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Injured as Result of Drinking',
               'section': 'Audit C - Positive',
               'column': 'injury_due_to_drinking',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Someone Been Concerned',
               'section': 'Audit C - Positive',
               'column': 'others_concerned',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AuditC Total Score',
               'section': 'Audit C - Positive',
               'column': 'full_audit_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'delirium_dementia_screening',
 'display': 'Delirium and dementia screening (current and legacy eras)',
 'source_displays': ['Delirium & Dementia Screening', 'Dementia Screening Form'],
 'elements': [{'label': 'Emergency Override',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'emergency_override',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Deferring due to Medical Emergency',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'deferred_for_medical_emergency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Pt has diagnosis of Delirium',
               'matches': [{'label': 'Pt has diagnosis of Delirium',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Delirium Diagnosis',
                            'section': 'Stage 1_Dementia Screening Asst'},
                           {'label': 'DS Delirium Diagnosis',
                            'section': 'Dementia Sreening Asst_Stage 1'}],
               'column': 'delirium_diagnosis',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Pt has diagnosis of Dementia',
               'matches': [{'label': 'Pt has diagnosis of Dementia',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Dementia Diagnosis',
                            'section': 'Stage 1_Dementia Screening Asst'},
                           {'label': 'DS Dementia Diagnosis',
                            'section': 'Dementia Sreening Asst_Stage 1'}],
               'column': 'dementia_diagnosis',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Pt been more forgetful last 12 Months',
               'matches': [{'label': 'Pt been more forgetful last 12 Months',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS More Forgetful',
                            'section': 'Stage 1_Dementia Screening Asst'},
                           {'label': 'DS More Forgetful',
                            'section': 'Dementia Sreening Asst_Stage 1'}],
               'column': 'more_forgetful_last_12_months',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Diagnosis of Delirium Outcome',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'delirium_outcome',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT Alertness',
               'matches': [{'label': '4AT Alertness',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Alertness', 'section': '4AT Checklist'}],
               'column': 'four_at_alertness',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT AMT4',
               'matches': [{'label': '4AT AMT4',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS AMT4', 'section': '4AT Checklist'}],
               'column': 'four_at_amt4',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT Attention',
               'matches': [{'label': '4AT Attention',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Attention', 'section': '4AT Checklist'}],
               'column': 'four_at_attention',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT Acute Change/Fluctuating Course',
               'matches': [{'label': '4AT Acute Change/Fluctuating Course',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Acute change or fluctuating course',
                            'section': '4AT Checklist'}],
               'column': 'four_at_acute_change',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT Outcome',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'four_at_outcome',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '4AT Score',
               'matches': [{'label': '4AT Score',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Total Score', 'section': '4AT Checklist'}],
               'column': 'four_at_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Problem with Cognitive Function',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'cognitive_function_problem',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Problem with Perception',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'perception_problem',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Changes in Physical Function',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'physical_function_change',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Changes in Social Behaviour',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'social_behaviour_change',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'More Forgetful, Reason not appropriate',
               'matches': [{'label': 'More Forgetful, Reason not appropriate',
                            'section': 'Delirium & Dementia Screening - Stage 1 of 2'},
                           {'label': 'DS Reason', 'section': 'Stage 1_Dementia Screening Asst'},
                           {'label': 'DS Reason', 'section': 'Dementia Sreening Asst_Stage 1'}],
               'column': 'screening_not_appropriate_reason',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Outcome',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_outcome',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Age',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_age',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Time',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_time',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Year',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_year',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS , Name of this place',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_place',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Date of Birth',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_date_of_birth',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Identification of Two Persons',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_identify_two_people',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Year of 1st World War',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_first_world_war_year',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Name of Present Monarch',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_monarch',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Counts backwards 20 -1',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_count_backwards',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Address Recall',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_address_recall',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'AMTS, Total Score',
               'section': 'Delirium & Dementia Screening - Stage 1 of 2',
               'column': 'amts_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'DS OPLS Referral',
               'matches': [{'label': 'DS OPLS Referral',
                            'section': 'Delirium & Dementia Screening - Stage 2 of 2'},
                           {'label': 'DS OPLS Referral',
                            'section': 'Stage 2_Dementia Screening Asst'},
                           {'label': 'DS OPLS Referral',
                            'section': 'Dementia Sreening Asst_Stage 2'}],
               'column': 'opls_referral',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'DS Cognitive Screening Test Result',
               'matches': [{'label': 'DS Cognitive Screening Test Result',
                            'section': 'Stage 2_Dementia Screening Asst'},
                           {'label': 'DS Cognitive Screening Test Result',
                            'section': 'Dementia Sreening Asst_Stage 2'}],
               'column': 'cognitive_screening_test_result',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'DS Collateral History',
               'matches': [{'label': 'DS Collateral History',
                            'section': 'Delirium & Dementia Screening - Stage 2 of 2'},
                           {'label': 'DS Collateral History',
                            'section': 'Stage 2_Dementia Screening Asst'},
                           {'label': 'DS Collateral History',
                            'section': 'Dementia Sreening Asst_Stage 2'}],
               'column': 'collateral_history',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'DS Referral Completed',
               'matches': [{'label': 'DS Referral Completed',
                            'section': 'Delirium & Dementia Screening - Stage 2 of 2'},
                           {'label': 'DS Referral Completed',
                            'section': 'Stage 2_Dementia Screening Asst'},
                           {'label': 'DS Referral Completed',
                            'section': 'Dementia Sreening Asst_Stage 2'}],
               'column': 'referral_completed',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'smoking_status',
 'display': 'Smoking status and cessation support',
 'source_displays': ['Smoking Status Form'],
 'elements': [{'label': 'SC Smoking Status',
               'section': 'Smoking Status Form',
               'column': 'smoking_status',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC Referral to Stop Smoking Service',
               'section': 'Smoking Status Form',
               'column': 'stop_smoking_referral',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC Nicotine Replacement Therapy',
               'section': 'Smoking Status Form',
               'column': 'nicotine_replacement_therapy',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC Very brief advice given',
               'section': 'Smoking Status Form',
               'column': 'very_brief_advice_given',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC Level dependence to Nicotine',
               'section': 'Smoking Status Form',
               'column': 'nicotine_dependence_level',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC NRT Supplied',
               'section': 'Smoking Status Form',
               'column': 'nrt_supplied',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'SC Referral Comments',
               'section': 'Smoking Status Form',
               'column': 'referral_comments',
               'kind': 'TEXT',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'frailty_score',
 'display': 'Rockwood clinical frailty score (both form eras)',
 'source_displays': ['Frailty Score', 'Frailty Screening Tool'],
 'elements': [{'label': 'Rockwood Frailty Score',
               'matches': [{'label': 'Rockwood Frailty Score', 'section': 'Frailty Score'},
                           {'label': 'Rockwood Frailty Score',
                            'section': 'Clinical Frailty Scale'}],
               'column': 'rockwood_frailty_score',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'adult_sepsis_screening',
 'display': 'Adult sepsis screening and SBAR escalation (both form eras)',
 'source_displays': ['Adult Sepsis Screening Tool', 'Adult Sepsis Nurse Screening'],
 'elements': [{'label': 'Sepsis - Red Flag',
               'section': 'Adult Sepsis Screening Tool',
               'column': 'red_flag',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Sepsis - Amber Flag',
               'section': 'Adult Sepsis Screening Tool',
               'column': 'amber_flag',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Sepsis - Escalate to Job Role',
               'section': 'Adult Sepsis Screening Tool',
               'column': 'escalate_to_job_role',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SEP Patient not for escalation in the event of det',
               'section': 'Adult Sepsis Nurse Screening Tools',
               'column': 'not_for_escalation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SEP Time of deterioration',
               'matches': [{'label': 'SEP Time of deterioration',
                            'section': 'Adult Sepsis Screening Tool'},
                           {'label': 'SEP Time of deterioration',
                            'section': 'Adult Sepsis Nurse Screening Tools'}],
               'column': 'screening_time_of_deterioration',
               'kind': 'DATE',
               'cardinality': 'single'},
              {'label': 'SEP Time of deterioration',
               'section': 'SBAR Escalation',
               'column': 'escalation_time_of_deterioration',
               'kind': 'DATE',
               'cardinality': 'single'},
              {'label': 'SEP Nurse',
               'section': 'Adult Sepsis Nurse Screening Tools',
               'column': 'nurse',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SEP Moderate risk criteria',
               'matches': [{'label': 'SEP Moderate risk criteria',
                            'section': 'Adult Sepsis Screening Tool'},
                           {'label': 'SEP Moderate risk criteria',
                            'section': 'Adult Sepsis Nurse Screening Tools'}],
               'column': 'moderate_risk_criteria',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'SEP High risk criteria',
               'matches': [{'label': 'SEP High risk criteria',
                            'section': 'Adult Sepsis Screening Tool'},
                           {'label': 'SEP High risk criteria',
                            'section': 'Adult Sepsis Nurse Screening Tools'}],
               'column': 'high_risk_criteria',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'SEP Name of doctor contacted',
               'section': 'Adult Sepsis Nurse Screening Tools',
               'column': 'doctor_contacted',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Escalation Required',
               'section': 'SBAR Escalation',
               'column': 'escalation_required',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SBAR Identify yourself',
               'section': 'SBAR Escalation',
               'column': 'identify_yourself',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Role',
               'section': 'SBAR Escalation',
               'column': 'role',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SBAR Recommendation',
               'section': 'SBAR Escalation',
               'column': 'recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SBAR Outline concern',
               'section': 'SBAR Escalation',
               'column': 'outline_concern',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Other clinical signs',
               'section': 'SBAR Escalation',
               'column': 'other_clinical_signs',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'SBAR Agreed plan',
               'section': 'SBAR Escalation',
               'column': 'agreed_plan',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Agree timeframe for review by team/CCOT/ART',
               'section': 'SBAR Escalation',
               'column': 'review_timeframe',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SBAR Ask what interventions you can do?',
               'section': 'SBAR Escalation',
               'column': 'interventions',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'SBAR Name of person spoken to',
               'section': 'SBAR Escalation',
               'column': 'person_spoken_to',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR State your clinical impression',
               'section': 'SBAR Escalation',
               'column': 'clinical_impression',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Additional information',
               'section': 'SBAR Escalation',
               'column': 'additional_information',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR What else do you need?',
               'section': 'SBAR Escalation',
               'column': 'what_else_needed',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'SBAR Rationale for not escalating',
               'section': 'SBAR Escalation',
               'column': 'not_escalating_rationale',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'swallowing_screen',
 'display': 'Swallowing screen',
 'source_displays': ['Swallowing Screen'],
 'elements': [{'label': 'Swallow screen complete',
               'section': 'SWALLOWING SCREEN',
               'column': 'swallow_screen_complete',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Swallow Screen Calc',
               'section': 'SWALLOWING SCREEN',
               'column': 'swallow_screen_calc',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Teaspoon of water Problem Encountered',
               'section': 'SWALLOWING SCREEN',
               'column': 'teaspoon_of_water_problem_encountered',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Sip of water Problem Encountered',
               'section': 'SWALLOWING SCREEN',
               'column': 'sip_of_water_problem_encountered',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Glass of water\xa0 Problem Encountered',
               'section': 'SWALLOWING SCREEN',
               'column': 'glass_of_water_problem_encountered',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Passed Swallow Screen Recommendations',
               'section': 'SWALLOWING SCREEN',
               'column': 'passed_swallow_screen_recommendations',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Failed Screen Recommendations',
               'section': 'SWALLOWING SCREEN',
               'column': 'failed_screen_recommendations',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Failed Swallow Screen Recommendations',
               'section': 'SWALLOWING SCREEN',
               'column': 'failed_swallow_screen_recommendations',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': '1st Teaspoon of Water Problem',
               'section': 'SWALLOWING SCREEN',
               'column': 'item_1st_teaspoon_of_water_problem',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Give sip of water Problem',
               'section': 'SWALLOWING SCREEN',
               'column': 'give_sip_of_water_problem',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Give glass water Problem',
               'section': 'SWALLOWING SCREEN',
               'column': 'give_glass_water_problem',
               'kind': 'CODED',
               'cardinality': 'multi'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'must',
 'display': 'Malnutrition Universal Screening Tool',
 'source_displays': ['Malnutrition Universal Screening Tool'],
 'elements': [{'label': 'Source of BMI',
               'section': 'MUST',
               'column': 'source_of_bmi',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Acute Disease Effect Score',
               'section': 'MUST',
               'column': 'acute_disease_effect_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Height/Length Measured',
               'section': 'MUST',
               'column': 'height_length_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Weight Measured',
               'section': 'MUST',
               'column': 'weight_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Weight Type',
               'section': 'MUST',
               'column': 'weight_type',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Overall Risk of Malnutrition',
               'section': 'MUST',
               'column': 'overall_risk_of_malnutrition',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'BMI',
               'section': 'MUST',
               'column': 'bmi',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Usual Weight Prior to Admission',
               'section': 'MUST',
               'column': 'usual_weight_prior_to_admission',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Weight Loss Score (MUST)',
               'section': 'MUST',
               'column': 'weight_loss_score_must',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': '% Weight Loss',
               'section': 'MUST',
               'column': 'weight_loss',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Patient Type (MUST)',
               'section': 'MUST',
               'column': 'patient_type_must',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Admission /Clinic Date (MUST)',
               'section': 'MUST',
               'column': 'admission_clinic_date_must',
               'kind': 'DATE',
               'cardinality': 'single'},
              {'label': 'Body Mass Index Measured',
               'section': 'MUST',
               'column': 'body_mass_index_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'IP Actions Plan (MUST)',
               'section': 'MUST',
               'column': 'ip_actions_plan_must',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Mid Upper Arm Circumference (cm)',
               'section': 'MUST',
               'column': 'mid_upper_arm_circumference_cm',
               'kind': 'NUMERIC',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'mood_rankin',
 'display': 'Mood screening and Modified Rankin Scale',
 'source_displays': ['Mood Screening Assessment/Modified Rankin Scale'],
 'elements': [{'label': 'Patient needs encouragement to do things',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_needs_encouragement_to_do_things',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient looks sad/miserable/depressed',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_looks_sad_miserable_depressed',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient seems withdrawn, lacks interest',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_seems_withdrawn_lacks_interest',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient seems agitated/restless/anxious',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_seems_agitated_restless_anxious',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient ever cry or seem weepy',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_ever_cry_or_seem_weepy',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient lethargic/reluctant to mobilise',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'patient_lethargic_reluctant_to_mobilise',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SDSS Total Score',
               'section': 'MOOD SCREENING ASSESSMENT',
               'column': 'sdss_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Modified Rankin Scale Assessment',
               'section': 'MODIFIED RANKINE SCALE',
               'column': 'modified_rankin_scale_assessment',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'nih_stroke_score',
 'display': 'NIH Stroke Score form layout',
 'source_displays': ['NIH Stroke Score'],
 'elements': [{'label': 'Level Conscious Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'level_conscious_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Level of Consciousness NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'level_of_consciousness_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Open/Close Eyes NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'open_close_eyes_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Open/Close Eyes Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'open_close_eyes_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Best Gaze NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'best_gaze_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Best Gaze Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'best_gaze_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Respond Month Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'respond_month_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Response Month/Age NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'response_month_age_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Facial Paresis NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'facial_paresis_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Facial Paresis Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'facial_paresis_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Motor Function Rt Arm NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_function_rt_arm_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Motor Rt Arm Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_rt_arm_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Motor Function Lt Arm NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_function_lt_arm_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Motor Lt Arm Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_lt_arm_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Visual Field Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'visual_field_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Visual Field Testing NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'visual_field_testing_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Motor Function Lt Leg NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_function_lt_leg_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Motor Function Rt Leg NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_function_rt_leg_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Motor Lt Leg Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_lt_leg_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Motor Rt Leg Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'motor_rt_leg_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Dysarthria NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'dysarthria_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Dysarthria Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'dysarthria_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Limb Ataxia NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'limb_ataxia_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Limb Ataxia Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'limb_ataxia_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Extinction Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'extinction_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Extinction/Inattention NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'extinction_inattention_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Best Language NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'best_language_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Best Language Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'best_language_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Sensory NIH Stroke Scale',
               'section': 'NIH STROKE SCORE',
               'column': 'sensory_nih_stroke_scale',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Sensory Score Stroke',
               'section': 'NIH STROKE SCORE',
               'column': 'sensory_score_stroke',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'NIH Stroke Score',
               'section': 'NIH STROKE SCORE',
               'column': 'nih_stroke_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Limb Ataxia NIH Stroke Interp',
               'section': 'NIH STROKE SCORE',
               'column': 'limb_ataxia_nih_stroke_interp',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'maternity_co_smoking',
 'display': 'Maternity carbon monoxide and smoking assessment',
 'source_displays': ['Maternity Carbon Monoxide / Smoking'],
 'carry_pregnancy_link': True,
 'elements': [{'label': 'Gestation days',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Gestation weeks',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'gestation_weeks',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Date',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'date',
               'kind': 'DATE',
               'cardinality': 'single'},
              {'label': 'Carbon Monoxide Reading Performed',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'carbon_monoxide_reading_performed',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MCO Smoking Status Mother',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'mco_smoking_status_mother',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Carbon Monoxide Reading',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'carbon_monoxide_reading',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'SC Referral to Stop Smoking Service',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'sc_referral_to_stop_smoking_service',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'SC Very brief advice given',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'sc_very_brief_advice_given',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Stop Smoking Service  Referral Required',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'stop_smoking_service_referral_required',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Stop Smoking Support Received this Preg',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'stop_smoking_support_received_this_preg',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Comments',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'comments',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'Reason Carbon Monoxide reading not performed',
               'section': 'MATERNITY CARBON MONOXIDE / SMOKING',
               'column': 'reason_carbon_monoxide_reading_not_performed',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'poms',
 'display': 'Post-Operative Morbidity Survey',
 'source_displays': ['Post-Operative Morbidity Survey (POMS)'],
 'elements': [{'label': 'POMS Cardiovascular',
               'column': 'poms_cardiovascular',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Cardiovascular',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Cardiovascular', 'section': 'POMS'}]},
              {'label': 'POMS Gastrointestinal',
               'column': 'poms_gastrointestinal',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Gastrointestinal',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Gastrointestinal', 'section': 'POMS'}]},
              {'label': 'POMS Haematological',
               'column': 'poms_haematological',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Haematological',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Haematological', 'section': 'POMS'}]},
              {'label': 'POMS Infectious',
               'column': 'poms_infectious',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Infectious',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Infectious', 'section': 'POMS'}]},
              {'label': 'POMS Neurological',
               'column': 'poms_neurological',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Neurological',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Neurological', 'section': 'POMS'}]},
              {'label': 'POMS Pain',
               'column': 'poms_pain',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Pain',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Pain', 'section': 'POMS'}]},
              {'label': 'POMS Pulmonary',
               'column': 'poms_pulmonary',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Pulmonary',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Pulmonary', 'section': 'POMS'}]},
              {'label': 'POMS Wound',
               'column': 'poms_wound',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Wound',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Wound', 'section': 'POMS'}]},
              {'label': 'POMS Renal',
               'column': 'poms_renal',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Renal',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'},
                           {'label': 'POMS Renal', 'section': 'POMS'}]},
              {'label': 'POMS Pain Comment',
               'column': 'poms_pain_comment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Pain Comment',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'}]},
              {'label': 'POMS Renal Comment',
               'column': 'poms_renal_comment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Renal Comment',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'}]},
              {'label': 'POMS Cardiovascular Comment',
               'column': 'poms_cardiovascular_comment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Cardiovascular Comment',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'}]},
              {'label': 'POMS Gastrointestinal Comment',
               'column': 'poms_gastrointestinal_comment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POMS Gastrointestinal Comment',
                            'section': 'POST OPERATIVE MORBIDITY SURVEY (POMS)'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'moca',
 'display': 'Montreal Cognitive Assessment',
 'source_displays': ['Montreal Cognitive Assessment (MOCA)'],
 'elements': [{'label': 'MOCA - Abstraction Points',
               'section': 'MOCA II',
               'column': 'moca_abstraction_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task One Points',
               'section': 'MOCA I',
               'column': 'moca_attention_task_one_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task Three Points',
               'section': 'MOCA I',
               'column': 'moca_attention_task_three_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task Two Points',
               'section': 'MOCA I',
               'column': 'moca_attention_task_two_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Fluency Points',
               'section': 'MOCA II',
               'column': 'moca_fluency_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Naming Points',
               'section': 'MOCA I',
               'column': 'moca_naming_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Orientation Points',
               'section': 'MOCA II',
               'column': 'moca_orientation_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Repeat Points',
               'section': 'MOCA II',
               'column': 'moca_repeat_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Visuospatial Points',
               'section': 'MOCA I',
               'column': 'moca_visuospatial_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Abstraction First',
               'section': 'MOCA II',
               'column': 'moca_abstraction_first',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Abstraction Second',
               'section': 'MOCA II',
               'column': 'moca_abstraction_second',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task One A',
               'section': 'MOCA I',
               'column': 'moca_attention_task_one_a',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task One B',
               'section': 'MOCA I',
               'column': 'moca_attention_task_one_b',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task Three',
               'section': 'MOCA I',
               'column': 'moca_attention_task_three',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Attention Task Two',
               'section': 'MOCA I',
               'column': 'moca_attention_task_two',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Camel',
               'section': 'MOCA I',
               'column': 'moca_camel',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - City',
               'section': 'MOCA II',
               'column': 'moca_city',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Contour',
               'section': 'MOCA I',
               'column': 'moca_contour',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Copy Cube',
               'section': 'MOCA I',
               'column': 'moca_copy_cube',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Date',
               'section': 'MOCA II',
               'column': 'moca_date',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Day',
               'section': 'MOCA II',
               'column': 'moca_day',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Fluency',
               'section': 'MOCA II',
               'column': 'moca_fluency',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Hands',
               'section': 'MOCA I',
               'column': 'moca_hands',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Lion',
               'section': 'MOCA I',
               'column': 'moca_lion',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Month',
               'section': 'MOCA II',
               'column': 'moca_month',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Numbers',
               'section': 'MOCA I',
               'column': 'moca_numbers',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Place',
               'section': 'MOCA II',
               'column': 'moca_place',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Repeat First',
               'section': 'MOCA II',
               'column': 'moca_repeat_first',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Repeat Second',
               'section': 'MOCA II',
               'column': 'moca_repeat_second',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Rhinoceros',
               'section': 'MOCA I',
               'column': 'moca_rhinoceros',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Unable to Assess',
               'section': 'MOCA I',
               'column': 'moca_unable_to_assess',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Visuospatial',
               'section': 'MOCA I',
               'column': 'moca_visuospatial',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Year',
               'section': 'MOCA II',
               'column': 'moca_year',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MOCA - Delayed Recall Points',
               'section': 'MOCA II',
               'column': 'moca_delayed_recall_points',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Total Score',
               'section': 'MOCA II',
               'column': 'moca_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'MOCA - Delayed Recall',
               'section': 'MOCA II',
               'column': 'moca_delayed_recall',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'MOCA - Administered',
               'section': 'MOCA II',
               'column': 'moca_administered',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'MOCA - Category Cue',
               'section': 'MOCA II',
               'column': 'moca_category_cue',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'MOCA - Memory First',
               'section': 'MOCA I',
               'column': 'moca_memory_first',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'MOCA - Memory Second',
               'section': 'MOCA I',
               'column': 'moca_memory_second',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'MOCA - Comments',
               'section': 'MOCA II',
               'column': 'moca_comments',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'MOCA - Multi Choice Cue',
               'section': 'MOCA II',
               'column': 'moca_multi_choice_cue',
               'kind': 'CODED',
               'cardinality': 'multi'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'waterlow',
 'display': 'Waterlow pressure-ulcer risk assessment',
 'source_displays': ['Waterlow Score'],
 'elements': [{'label': 'Major Surgery/Trauma (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'major_surgery_trauma_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Skin Type and Visual Risk Areas',
               'section': 'WATERLOW SCORE',
               'column': 'skin_type_and_visual_risk_areas',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Appetite (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'appetite_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Continence (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'continence_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Neurological Deficits (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'neurological_deficits_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Recent Weight Loss (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'recent_weight_loss_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Mobility (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'mobility_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Tissue Malnutrition (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'tissue_malnutrition_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Medication (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'medication_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Sex/Age (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'sex_age_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Waterlow Score Result',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_score_result',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Build for Height (Waterlow Score NUH)',
               'section': 'WATERLOW SCORE',
               'column': 'build_for_height_waterlow_score_nuh',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Weight Loss (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'weight_loss_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Age (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'age_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Build/height for weight',
               'section': 'WATERLOW SCORE',
               'column': 'build_height_for_weight',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Continence',
               'section': 'WATERLOW SCORE',
               'column': 'continence',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Risk areas visual skin type',
               'section': 'WATERLOW SCORE',
               'column': 'risk_areas_visual_skin_type',
               'kind': 'CODED',
               'cardinality': 'multi'},
              {'label': 'Sex (Waterlow Score)',
               'section': 'WATERLOW SCORE',
               'column': 'sex_waterlow_score',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Appetite',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_appetite',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Major Surgery or trauma',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_major_surgery_or_trauma',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Neurological deficit',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_neurological_deficit',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Tissue Malnutrition',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_tissue_malnutrition',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow mobility',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_mobility',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Total Score',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Waterlow Total Score - Risk',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_total_score_risk',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Waterlow Medications',
               'section': 'WATERLOW SCORE',
               'column': 'waterlow_medications',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Height/Length Measured',
               'section': 'WATERLOW SCORE',
               'column': 'height_length_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Comments (Waterlow)',
               'section': 'WATERLOW SCORE',
               'column': 'comments_waterlow',
               'kind': 'TEXT',
               'cardinality': 'single'},
              {'label': 'Weight (kg)',
               'section': 'WATERLOW SCORE',
               'column': 'weight_kg',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'zzzHeight (metres)',
               'section': 'WATERLOW SCORE',
               'column': 'zzzheight_metres',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Weight Measured',
               'section': 'WATERLOW SCORE',
               'column': 'weight_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'},
              {'label': 'Body Mass Index Measured',
               'section': 'WATERLOW SCORE',
               'column': 'body_mass_index_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'infection_prevention',
 'display': 'Infection prevention admission assessment',
 'source_displays': ['Infection Prevention Assessment'],
 'elements': [{'label': 'MRSA Screening on Admission',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'mrsa_screening_on_admission',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Healthcare Overseas in Last 12 Months',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'healthcare_overseas_in_last_12_months',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Known to Have Resistant Organisms',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'known_to_have_resistant_organisms',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Resistant Organism Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'resistant_organism_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient has diarrhoea',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'patient_has_diarrhoea',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Evidence of Pulmonary Tuberculosis',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'evidence_of_pulmonary_tuberculosis',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Tuberculosis Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'tuberculosis_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Viral Respiratory Tract Infection',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'viral_respiratory_tract_infection',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Inpatient in UK Hosp in Last 12 Months',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'inpatient_in_uk_hosp_in_last_12_months',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'CRO Screening Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'cro_screening_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient has rash',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'patient_has_rash',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Patient has Vomiting',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'patient_has_vomiting',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Enteric Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'enteric_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Viral RTI Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'viral_rti_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'Rash Recommendation',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'rash_recommendation',
               'kind': 'CODED',
               'cardinality': 'single'},
              {'label': 'MRSA Reason Not Taken',
               'section': 'INFECTION PREVENTION ASSESSMENT',
               'column': 'mrsa_reason_not_taken',
               'kind': 'CODED',
               'cardinality': 'single'}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'vte_risk_assessment',
 'display': 'Adult VTE risk assessment',
 'source_displays': ['VTE Risk Assessment_FCST', 'VTE Assessment'],
 'elements': [{'label': 'VTE Override',
               'column': 'vte_override',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Override', 'section': 'MOBILITY-FCST'}]},
              {'label': 'Mobility - All Patients',
               'column': 'mobility_all_patients',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Mobility - All Patients', 'section': 'MOBILITY-FCST'}]},
              {'label': 'VTE Override Reason',
               'column': 'vte_override_reason',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Override Reason', 'section': 'MOBILITY-FCST'}]},
              {'label': 'Thrombosis Risk - Patient Related',
               'column': 'thrombosis_risk_patient_related',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Thrombosis Risk - Patient Related',
                            'section': 'SURG/MED-FCST'}]},
              {'label': 'Bleeding Risk - Admission Related',
               'column': 'bleeding_risk_admission_related',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Bleeding Risk - Admission Related',
                            'section': 'SURG/MED-FCST'}]},
              {'label': 'Thrombosis Risk - Admission Related',
               'column': 'thrombosis_risk_admission_related',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Thrombosis Risk - Admission Related',
                            'section': 'SURG/MED-FCST'}]},
              {'label': 'Bleeding Risk - Patient Related',
               'column': 'bleeding_risk_patient_related',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Bleeding Risk - Patient Related',
                            'section': 'SURG/MED-FCST'}]},
              {'label': 'VTE Risk',
               'column': 'vte_risk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Risk', 'section': 'SURG/MED-FCST'}]},
              {'label': 'Bleeding Risk',
               'column': 'bleeding_risk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Bleeding Risk', 'section': 'SURG/MED-FCST'}]},
              {'label': 'VTE Prophylaxis Calc',
               'column': 'vte_prophylaxis_calc',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Prophylaxis Calc', 'section': 'MOBILITY-FCST'}]},
              {'label': 'VTE Prophylaxis Prescribed',
               'column': 'vte_prophylaxis_prescribed_surg_med_fcst',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Prophylaxis Prescribed', 'section': 'SURG/MED-FCST'}]},
              {'label': 'zzzVTE Risk Intrp 5',
               'column': 'zzzvte_risk_intrp_5',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'zzzVTE Risk Intrp 5', 'section': 'SURG/MED-FCST'}]},
              {'label': 'zzzVTE Risk Intrp 1',
               'column': 'zzzvte_risk_intrp_1',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'zzzVTE Risk Intrp 1', 'section': 'SURG/MED-FCST'}]},
              {'label': 'zzzVTE Risk Intrp 2',
               'column': 'zzzvte_risk_intrp_2',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'zzzVTE Risk Intrp 2', 'section': 'SURG/MED-FCST'}]},
              {'label': 'zzzVTE Risk Intrp 3',
               'column': 'zzzvte_risk_intrp_3',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'zzzVTE Risk Intrp 3', 'section': 'SURG/MED-FCST'}]},
              {'label': 'zzzVTE Risk Intrp 4',
               'column': 'zzzvte_risk_intrp_4',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'zzzVTE Risk Intrp 4', 'section': 'SURG/MED-FCST'}]},
              {'label': 'VTE Prophylaxis Prescribed',
               'column': 'vte_prophylaxis_prescribed_prescribing',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Prophylaxis Prescribed', 'section': 'PRESCRIBING'}]},
              {'label': 'Obstetric VTE Risk Total v2',
               'column': 'obstetric_vte_risk_total_v2',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Obstetric VTE Risk Total v2',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Person Completing Form (VTE)',
               'column': 'person_completing_form_vte',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Person Completing Form (VTE)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Height/Length Measured',
               'column': 'height_length_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Height/Length Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Maternity VTE',
               'column': 'maternity_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE', 'section': 'MOBILITY-FCST'}]},
              {'label': 'Obstetric VTE Risk Assessment Type',
               'column': 'obstetric_vte_risk_assessment_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Obstetric VTE Risk Assessment Type',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Smoker',
               'column': 'smoker',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Smoker', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Previous VTE',
               'column': 'previous_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Previous VTE', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Current Systemic Infection',
               'column': 'current_systemic_infection',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Current Systemic Infection',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Gross Varicose Veins',
               'column': 'gross_varicose_veins',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Gross Varicose Veins',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'OHSS (Overian Hyperstimulation Syndrome)',
               'column': 'ohss_overian_hyperstimulation_syndrome',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'OHSS (Overian Hyperstimulation Syndrome)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Family History of VTE',
               'column': 'family_history_of_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Family History of VTE',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Hyperemesis',
               'column': 'hyperemesis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Hyperemesis', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Medical Comorbidities',
               'column': 'medical_comorbidities',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Medical Comorbidities',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'VTE Known Thrombophilia',
               'column': 'vte_known_thrombophilia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Known Thrombophilia',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Ob VTE Obesity',
               'column': 'ob_vte_obesity',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Ob VTE Obesity',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Patient at Risk of VTE',
               'column': 'patient_at_risk_of_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient at Risk of VTE',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Weight Measured',
               'column': 'weight_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Weight Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Body Mass Index Measured',
               'column': 'body_mass_index_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Body Mass Index Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Multiple Pregnancy (Twins or more)',
               'column': 'multiple_pregnancy_twins_or_more',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Multiple Pregnancy (Twins or more)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Age>35 / Parity >3',
               'column': 'age_35_parity_3',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Age>35 / Parity >3',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Dehydration/Reduced Immobility/ART/IVF',
               'column': 'dehydration_reduced_immobility_art_ivf',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Dehydration/Reduced Immobility/ART/IVF',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Surg procedure in this preg or <=6 weeks',
               'column': 'surg_procedure_in_this_preg_or_6_weeks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Surg procedure in this preg or <=6 weeks',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Pre-eclampsia',
               'column': 'pre_eclampsia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pre-eclampsia', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Elective Caesarean Section',
               'column': 'elective_caesarean_section',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Elective Caesarean Section',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Mid-Cavity or Rotational Forceps',
               'column': 'mid_cavity_or_rotational_forceps',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Mid-Cavity or Rotational Forceps',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Caesarean Section in Labour',
               'column': 'caesarean_section_in_labour',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Caesarean Section in Labour',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Still birth this pregnancy',
               'column': 'still_birth_this_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Still birth this pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Preterm birth this pregnancy',
               'column': 'preterm_birth_this_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Preterm birth this pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Age 36 or more',
               'column': 'age_36_or_more',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Age 36 or more',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Paraplegia',
               'column': 'paraplegia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Paraplegia', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Parity 3 or more',
               'column': 'parity_3_or_more',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Parity 3 or more',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Surgical procedure within last 6 weeks',
               'column': 'surgical_procedure_within_last_6_weeks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Surgical procedure within last 6 weeks',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Dehydration / Reduced Mobility/ ART/IVF',
               'column': 'dehydration_reduced_mobility_art_ivf',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Dehydration / Reduced Mobility/ ART/IVF',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Pre-eclampsia in this pregnancy',
               'column': 'pre_eclampsia_in_this_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pre-eclampsia in this pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Telephone Number',
               'column': 'telephone_number',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Telephone Number',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'VTE Reason Prophylaxis Not Prescribed',
               'column': 'vte_reason_prophylaxis_not_prescribed_prescribing',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Reason Prophylaxis Not Prescribed',
                            'section': 'PRESCRIBING'}]},
              {'label': 'Multiple Pregnancy',
               'column': 'multiple_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Multiple Pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'PPH (1 litre or more and/or transfusion)',
               'column': 'pph_1_litre_or_more_and_or_transfusion',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PPH (1 litre or more and/or transfusion)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Prolonged Labour (over 24 hours)',
               'column': 'prolonged_labour_over_24_hours',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Prolonged Labour (over 24 hours)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'PPH 1 litre or more or transfusion',
               'column': 'pph_1_litre_or_more_or_transfusion',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PPH 1 litre or more or transfusion',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Labour 24 hours or more',
               'column': 'labour_24_hours_or_more',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Labour 24 hours or more',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'BMI',
               'column': 'bmi',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'BMI', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'VTE Reason Prophylaxis Not Prescribed',
               'column': 'vte_reason_prophylaxis_not_prescribed_surg_med_fcst',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Reason Prophylaxis Not Prescribed',
                            'section': 'SURG/MED-FCST'}]},
              {'label': 'Appropriate Thrombo for Dr to prescribe',
               'column': 'appropriate_thrombo_for_dr_to_prescribe',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Appropriate Thrombo for Dr to prescribe',
                            'section': 'OBSTETRICS VTE TREATMENT PLAN'}]},
              {'label': 'Contraindication to LMWH or Heparin',
               'column': 'contraindication_to_lmwh_or_heparin',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Contraindication to LMWH or Heparin',
                            'section': 'OBSTETRICS VTE TREATMENT PLAN'}]},
              {'label': '(System Use)VTE Assessment Condition 2',
               'column': 'system_use_vte_assessment_condition_2',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': '(System Use)VTE Assessment Condition 2',
                            'section': 'VTE ASSESSMENT P1'}]},
              {'label': 'VTE Surgical',
               'column': 'vte_surgical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Surgical', 'section': 'VTE ASSESSMENT P1'}]},
              {'label': '(System Use)VTE Assessment Status',
               'column': 'system_use_vte_assessment_status',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': '(System Use)VTE Assessment Status',
                            'section': 'VTE ASSESSMENT P1'}]},
              {'label': 'VTE Mobility',
               'column': 'vte_mobility',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Mobility', 'section': 'VTE ASSESSMENT P1'}]},
              {'label': '(System Use)VTE Assessment Condition 1',
               'column': 'system_use_vte_assessment_condition_1',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': '(System Use)VTE Assessment Condition 1',
                            'section': 'VTE ASSESSMENT P1'}]},
              {'label': 'VTE Other Factor(s)',
               'column': 'vte_other_factor_s',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'VTE Other Factor(s)', 'section': 'VTE ASSESSMENT P1'}]},
              {'label': 'VTE Contra Indications',
               'column': 'vte_contra_indications',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Contra Indications', 'section': 'VTE ASSESSMENT P1'}]},
              {'label': 'VTE consider contra indications',
               'column': 'vte_consider_contra_indications',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE consider contra indications',
                            'section': 'VTE ASSESSMENT P2'}]},
              {'label': 'VTE intermittent pneumatic',
               'column': 'vte_intermittent_pneumatic',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE intermittent pneumatic',
                            'section': 'VTE ASSESSMENT P2'}]},
              {'label': 'VTE Verbal Information',
               'column': 'vte_verbal_information',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Verbal Information', 'section': 'VTE ASSESSMENT P3'}]},
              {'label': 'VTE Written Information',
               'column': 'vte_written_information',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Written Information', 'section': 'VTE ASSESSMENT P3'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'adult_capacity_assessment',
 'display': 'Adult capacity admission assessment',
 'source_displays': ['Adult Capacity Admission Assessment'],
 'elements': [{'label': 'Patient has capacity to consent',
               'column': 'patient_has_capacity_to_consent',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient has capacity to consent',
                            'section': 'ADULT CAPACITY ADMISSION ASSESSMENT'}]},
              {'label': 'Consider DOLS review reference text',
               'column': 'consider_dols_review_reference_text',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Consider DOLS review reference text',
                            'section': 'ADULT CAPACITY ADMISSION ASSESSMENT'}]},
              {'label': 'Capacity Assessment Calculation',
               'column': 'capacity_assessment_calculation',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Capacity Assessment Calculation',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Impairment/Disturbance of Mind or Brain',
               'column': 'impairment_disturbance_of_mind_or_brain',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Impairment/Disturbance of Mind or Brain',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Capacity Assessment Outcome',
               'column': 'capacity_assessment_outcome',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Capacity Assessment Outcome',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Might The Impairement Affect Decision',
               'column': 'might_the_impairement_affect_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Might The Impairement Affect Decision',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Additional Support to Help Make Decision',
               'column': 'additional_support_to_help_make_decision',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Additional Support to Help Make Decision',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Patient Understands Decision to be Made',
               'column': 'patient_understands_decision_to_be_made',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Understands Decision to be Made',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Supporting evidence',
               'column': 'supporting_evidence',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Supporting evidence',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Patient Retains Info to Make Decision',
               'column': 'patient_retains_info_to_make_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Retains Info to Make Decision',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Patient Weighs Info to Make Decision',
               'column': 'patient_weighs_info_to_make_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Weighs Info to Make Decision',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]},
              {'label': 'Able To Communicate Decision by Any Mean',
               'column': 'able_to_communicate_decision_by_any_mean',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Able To Communicate Decision by Any Mean',
                            'section': 'MENTAL CAPACITY ASSESSMENT'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'ed_anderson_pressure_risk',
 'display': 'ED Anderson pressure-area risk assessment',
 'source_displays': ['ED Anderson Tool'],
 'elements': [{'label': 'AT_Is the patient fit to sit?',
               'column': 'at_is_the_patient_fit_to_sit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Is the patient fit to sit?',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Datex completed',
               'column': 'at_datex_completed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Datex completed',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Pressure area assessed - no damage identified',
               'column': 'at_pressure_area_assessed_no_damage_identified',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Pressure area assessed - no damage identified',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Total Score',
               'column': 'at_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Total Score',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Rehydration',
               'column': 'at_rehydration',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Rehydration',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_70 years of age',
               'column': 'at_70_years_of_age',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_70 years of age',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Actions Taken to reduce Pressure Ulcer Risk',
               'column': 'at_actions_taken_to_reduce_pressure_ulcer_risk',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Actions Taken to reduce Pressure Ulcer Risk',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Immobile',
               'column': 'at_immobile',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Immobile',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Limb mobility',
               'column': 'at_limb_mobility',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Limb mobility',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Incontinent',
               'column': 'at_incontinent',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Incontinent',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_6. Sacrum/Coccyx',
               'column': 'at_6_sacrum_coccyx',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_6. Sacrum/Coccyx',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Datix Reference',
               'column': 'at_datix_reference',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Datix Reference',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_7. Buttocks',
               'column': 'at_7_buttocks',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_7. Buttocks',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Other',
               'column': 'at_other',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Other',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Nutritionally',
               'column': 'at_nutritionally',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Nutritionally',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_15. Other',
               'column': 'at_15_other',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_15. Other',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Wounds',
               'column': 'at_wounds',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Wounds',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Length of time on the floor',
               'column': 'at_length_of_time_on_the_floor',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Length of time on the floor',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Reduced level of consciousness',
               'column': 'at_reduced_level_of_consciousness',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Reduced level of consciousness',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_12. Heel',
               'column': 'at_12_heel',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_12. Heel',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Orthopaedic',
               'column': 'at_orthopaedic',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Orthopaedic',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_5. Elbow',
               'column': 'at_5_elbow',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_5. Elbow',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_Moisture Lesion',
               'column': 'at_moisture_lesion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AT_Moisture Lesion',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]},
              {'label': 'AT_13. Foot (Lateral aspect)',
               'column': 'at_13_foot_lateral_aspect',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AT_13. Foot (Lateral aspect)',
                            'section': 'PRESSURE AREA RISK ASSESSMENT SCREENING TOOL'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'mental_health_liaison_assessment',
 'display': 'Mental-health liaison assessment (current and RAID eras)',
 'source_displays': ['Mental Health Liaison Service Assessment Form', 'RAID Assessment Form'],
 'elements': [{'label': 'RAID Assessment',
               'column': 'raid_assessment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Assessment', 'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Assessment', 'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Assessment Commenced Date and Time',
               'column': 'raid_assessment_commenced_date_and_time',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Assessment Commenced Date and Time',
                            'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Assessment Commenced Date and Time',
                            'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Intervention Picklist',
               'column': 'raid_intervention_picklist',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'RAID Intervention Picklist', 'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Intervention Picklist', 'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Location of Patient',
               'column': 'raid_location_of_patient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Location of Patient', 'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Location of Patient', 'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Discharge Destination List',
               'column': 'raid_discharge_destination_list',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Discharge Destination List',
                            'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Discharge Destination List',
                            'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Target Response Time Met',
               'column': 'raid_target_response_time_met',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Target Response Time Met', 'section': 'MHLS ASSESSMENT'},
                           {'label': 'RAID Target Response Time Met',
                            'section': 'RAID ASSESSMENT'}]},
              {'label': 'RAID Psycho Active Medication',
               'column': 'raid_psycho_active_medication',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'RAID Psycho Active Medication', 'section': 'RAID ASSESSMENT'},
                           {'label': 'RAID Psycho Active Medication',
                            'section': 'MHLS ASSESSMENT'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'enhanced_care_risk',
 'display': 'Enhanced care risk assessment',
 'source_displays': ['Enhanced Care Risk Assessment Tool'],
 'elements': [{'label': 'ECR risk of falls',
               'column': 'ecr_risk_of_falls',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ECR risk of falls', 'section': 'RISK OF FALLS'}]},
              {'label': 'ECR confusion/delirium/neuro symptoms.',
               'column': 'ecr_confusion_delirium_neuro_symptoms',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ECR confusion/delirium/neuro symptoms.',
                            'section': 'CONFUSION/DELIRIUM/NEURO SYMPTOMS/MENTAL '
                                       'ILLNESS/THREATENING OR AGGRESSIVE'}]},
              {'label': 'ECR self-harm/suicide',
               'column': 'ecr_self_harm_suicide',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ECR self-harm/suicide', 'section': 'SELF-HARM/SUICIDE'}]},
              {'label': 'ECR risk of patient becoming lost',
               'column': 'ecr_risk_of_patient_becoming_lost',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ECR risk of patient becoming lost',
                            'section': 'RISK OF PATIENT BECOMING LOST OR ABSCONDING'}]},
              {'label': 'ECR treatment/dependency, other clin iss',
               'column': 'ecr_treatment_dependency_other_clin_iss',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ECR treatment/dependency, other clin iss',
                            'section': 'TREATMENT AND DEPENDENCY, OTHER CLINICAL ISSUES'}]},
              {'label': 'Enhanced Care Risk Score',
               'column': 'enhanced_care_risk_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Enhanced Care Risk Score', 'section': 'LEVEL OF CARE'}]},
              {'label': 'Level of Care',
               'column': 'level_of_care',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Level of Care', 'section': 'LEVEL OF CARE'}]},
              {'label': 'Enhanced Care',
               'column': 'enhanced_care',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Enhanced Care', 'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 4',
               'column': 'ecr_interventions_level_4',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'ECR interventions level 4', 'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 3',
               'column': 'ecr_interventions_level_3',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'ECR interventions level 3', 'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 4 comments',
               'column': 'ecr_interventions_level_4_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'ECR interventions level 4 comments',
                            'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 3 comments',
               'column': 'ecr_interventions_level_3_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'ECR interventions level 3 comments',
                            'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 2',
               'column': 'ecr_interventions_level_2',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'ECR interventions level 2', 'section': 'LEVEL OF CARE'}]},
              {'label': 'ECR interventions level 2 comments',
               'column': 'ecr_interventions_level_2_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'ECR interventions level 2 comments',
                            'section': 'LEVEL OF CARE'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'resuscitation_plan',
 'display': 'Resuscitation plan and consultant validation',
 'source_displays': ['Resuscitation Plan', 'Resuscitation Plan - Consultant Validation Only'],
 'elements': [{'label': 'TEP Approving Senior',
               'column': 'tep_approving_senior',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Approving Senior', 'section': 'PLAN'}]},
              {'label': 'TEP Patient for CPR',
               'column': 'tep_patient_for_cpr',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Patient for CPR', 'section': 'PLAN'}]},
              {'label': 'TEP Age Group',
               'column': 'tep_age_group',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Age Group', 'section': 'PLAN'}]},
              {'label': 'TEP Countersigning Senior',
               'column': 'tep_countersigning_senior',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Countersigning Senior', 'section': 'PLAN'},
                           {'label': 'TEP Countersigning Senior',
                            'section': 'RESUSCITATION PLAN'}]},
              {'label': 'TEP Current Venue of Care',
               'column': 'tep_current_venue_of_care',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Current Venue of Care', 'section': 'PLAN'}]},
              {'label': 'TEP Reason(s)',
               'column': 'tep_reason_s',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'TEP Reason(s)', 'section': 'PLAN'}]},
              {'label': 'TEP Existing Community Instruction',
               'column': 'tep_existing_community_instruction',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Existing Community Instruction', 'section': 'PLAN'}]},
              {'label': 'TEP Discussed With',
               'column': 'tep_discussed_with',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'TEP Discussed With', 'section': 'PLAN'}]},
              {'label': 'Capacity Assessment Calculation',
               'column': 'capacity_assessment_calculation',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Capacity Assessment Calculation',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'Impairment/Disturbance of Mind or Brain',
               'column': 'impairment_disturbance_of_mind_or_brain',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Impairment/Disturbance of Mind or Brain',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'Capacity Assessment Outcome',
               'column': 'capacity_assessment_outcome',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Capacity Assessment Outcome', 'section': 'MENTAL CAPACITY'}]},
              {'label': 'TEP Review',
               'column': 'tep_review',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Review', 'section': 'RESUSCITATION PLAN'}]},
              {'label': 'TEP Period Valid',
               'column': 'tep_period_valid',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Period Valid', 'section': 'RESUSCITATION PLAN'}]},
              {'label': 'TEP Details of Discussion(s)',
               'column': 'tep_details_of_discussion_s',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Details of Discussion(s)', 'section': 'PLAN'}]},
              {'label': "TEP Patient's Priorities",
               'column': 'tep_patient_s_priorities',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': "TEP Patient's Priorities", 'section': 'PLAN'}]},
              {'label': 'TEP Suitable Treatments on Ward',
               'column': 'tep_suitable_treatments_on_ward',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Suitable Treatments on Ward', 'section': 'PLAN'}]},
              {'label': 'TEP Advanced Directive/Decision',
               'column': 'tep_advanced_directive_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Advanced Directive/Decision', 'section': 'PLAN'}]},
              {'label': 'TEP Proxy',
               'column': 'tep_proxy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Proxy', 'section': 'PLAN'}]},
              {'label': 'TEP Mental Capacity',
               'column': 'tep_mental_capacity',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Mental Capacity', 'section': 'PLAN'}]},
              {'label': 'TEP Supporting Information',
               'column': 'tep_supporting_information',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Supporting Information', 'section': 'PLAN'}]},
              {'label': 'Might The Impairement Affect Decision',
               'column': 'might_the_impairement_affect_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Might The Impairement Affect Decision',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'Additional Support to Help Make Decision',
               'column': 'additional_support_to_help_make_decision',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Additional Support to Help Make Decision',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'Patient Understands Decision to be Made',
               'column': 'patient_understands_decision_to_be_made',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Understands Decision to be Made',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'TEP Details of Senior Review',
               'column': 'tep_details_of_senior_review',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Details of Senior Review',
                            'section': 'RESUSCITATION PLAN'}]},
              {'label': 'TEP Details of Community Instruction',
               'column': 'tep_details_of_community_instruction',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Details of Community Instruction', 'section': 'PLAN'}]},
              {'label': 'TEP Suitable Treatments in Critical Care',
               'column': 'tep_suitable_treatments_in_critical_care',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'TEP Suitable Treatments in Critical Care',
                            'section': 'PLAN'}]},
              {'label': 'TEP Reason(s) for Not Discussing',
               'column': 'tep_reason_s_for_not_discussing',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Reason(s) for Not Discussing', 'section': 'PLAN'}]},
              {'label': 'Supporting evidence',
               'column': 'supporting_evidence',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Supporting evidence', 'section': 'MENTAL CAPACITY'}]},
              {'label': 'Patient Retains Info to Make Decision',
               'column': 'patient_retains_info_to_make_decision',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Retains Info to Make Decision',
                            'section': 'MENTAL CAPACITY'}]},
              {'label': 'TEP Legal Authority Person',
               'column': 'tep_legal_authority_person',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Legal Authority Person', 'section': 'PLAN'}]},
              {'label': 'TEP Legal Authority Type',
               'column': 'tep_legal_authority_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Legal Authority Type', 'section': 'PLAN'}]},
              {'label': 'TEP Discussed With (Other)',
               'column': 'tep_discussed_with_other',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'TEP Discussed With (Other)', 'section': 'PLAN'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'palliative_care_holistic_assessment',
 'display': 'Palliative care holistic assessment',
 'source_displays': ['Palliative Care Holistic Assessment'],
 'elements': [{'label': 'PAL - Seen by 1',
               'column': 'pal_seen_by_1',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Seen by 1', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Keyworker',
               'column': 'pal_keyworker',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Keyworker', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Team',
               'column': 'pal_team',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Team', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Patient Location',
               'column': 'pal_patient_location',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Patient Location', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Outcome of visit',
               'column': 'pal_outcome_of_visit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Outcome of visit', 'section': 'ACTIONS/OUTCOMES'}]},
              {'label': 'PAL - Preferred Place of Care',
               'column': 'pal_preferred_place_of_care',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Preferred Place of Care',
                            'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - Preferred place of death',
               'column': 'pal_preferred_place_of_death',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Preferred place of death',
                            'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - DNAR',
               'column': 'pal_dnar',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - DNAR', 'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - Overall Summary',
               'column': 'pal_overall_summary',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Overall Summary', 'section': 'ACTIONS/OUTCOMES'}]},
              {'label': 'PAL - Advance Care plan',
               'column': 'pal_advance_care_plan',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'PAL - Advance Care plan',
                            'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'Pal - Team Contact Details',
               'column': 'pal_team_contact_details',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pal - Team Contact Details', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Actions for Palliative Care team',
               'column': 'pal_actions_for_palliative_care_team',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Actions for Palliative Care team',
                            'section': 'ACTIONS/OUTCOMES'}]},
              {'label': 'PAL - Recommendations',
               'column': 'pal_recommendations',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Recommendations', 'section': 'ACTIONS/OUTCOMES'}]},
              {'label': 'PAL - Physical',
               'column': 'pal_physical',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Physical', 'section': 'ASSESSMENT'}]},
              {'label': 'PAL - Performance status Phase',
               'column': 'pal_performance_status_phase',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Performance status Phase',
                            'section': 'OUTCOMES & MEASURES'}]},
              {'label': 'PAL - Phase Score',
               'column': 'pal_phase_score',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Phase Score',
                            'section': 'PERFORMANCE STATUS - PHASE STATUS'}]},
              {'label': 'PAL - Performance status AKPS',
               'column': 'pal_performance_status_akps',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Performance status AKPS',
                            'section': 'OUTCOMES & MEASURES'}]},
              {'label': 'PAL - AKPS Score',
               'column': 'pal_akps_score',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - AKPS Score',
                            'section': 'PERFORMANCE STATUS - AKPS SCORE'}]},
              {'label': 'PAL - Social',
               'column': 'pal_social',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Social', 'section': 'ASSESSMENT'}]},
              {'label': 'PAL - Psychological',
               'column': 'pal_psychological',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Psychological', 'section': 'ASSESSMENT'}]},
              {'label': 'PAL - Spiritual',
               'column': 'pal_spiritual',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Spiritual', 'section': 'ASSESSMENT'}]},
              {'label': 'PAL - TEP',
               'column': 'pal_tep',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - TEP', 'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - Relationship',
               'column': 'pal_relationship',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'PAL - Relationship', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Name',
               'column': 'pal_name',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'PAL - Name', 'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Performance status IPOS',
               'column': 'pal_performance_status_ipos',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Performance status IPOS',
                            'section': 'OUTCOMES & MEASURES'}]},
              {'label': 'PAL - IPOS Q2  Pain',
               'column': 'pal_ipos_q2_pain',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Pain',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Weak',
               'column': 'pal_ipos_q2_weak',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Weak',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Poor appetite',
               'column': 'pal_ipos_q2_poor_appetite',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Poor appetite',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Poor mobility',
               'column': 'pal_ipos_q2_poor_mobility',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Poor mobility',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Nausea',
               'column': 'pal_ipos_q2_nausea',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Nausea',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Shortnes Breath',
               'column': 'pal_ipos_q2_shortnes_breath',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Shortnes Breath',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Vomiting',
               'column': 'pal_ipos_q2_vomiting',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Vomiting',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Constipation',
               'column': 'pal_ipos_q2_constipation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Constipation',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Drowsiness',
               'column': 'pal_ipos_q2_drowsiness',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Drowsiness',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2  Dry  Mouth',
               'column': 'pal_ipos_q2_dry_mouth',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2  Dry  Mouth',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Total Score Q3 to 9',
               'column': 'pal_ipos_total_score_q3_to_9',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Total Score Q3 to 9',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q7 Pat Share Feelings',
               'column': 'pal_ipos_q7_pat_share_feelings',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q7 Pat Share Feelings',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q6 Pat Felt at Peace',
               'column': 'pal_ipos_q6_pat_felt_at_peace',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q6 Pat Felt at Peace',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q8 Pat Had Enough Info',
               'column': 'pal_ipos_q8_pat_had_enough_info',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q8 Pat Had Enough Info',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q4 Family Worried',
               'column': 'pal_ipos_q4_family_worried',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q4 Family Worried',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q3 Pat Worried',
               'column': 'pal_ipos_q3_pat_worried',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q3 Pat Worried',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q5 Pat Depressed',
               'column': 'pal_ipos_q5_pat_depressed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q5 Pat Depressed',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q9 Pat Problems Addressed',
               'column': 'pal_ipos_q9_pat_problems_addressed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q9 Pat Problems Addressed',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - Key Worker Contact Details',
               'column': 'pal_key_worker_contact_details',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Key Worker Contact Details',
                            'section': 'CONTACT DETAILS'}]},
              {'label': 'PAL - Seen by 2',
               'column': 'pal_seen_by_2',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Seen by 2', 'section': 'CONTACT DETAILS'}]},
              {'label': 'Advance Care Plan Comments',
               'column': 'advance_care_plan_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Advance Care Plan Comments',
                            'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - IPOS Total Score Q1 to 3',
               'column': 'pal_ipos_total_score_q1_to_3',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Total Score Q1 to 3',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2 Other Symptoms 1',
               'column': 'pal_ipos_q2_other_symptoms_1',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2 Other Symptoms 1',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS All Total Score',
               'column': 'pal_ipos_all_total_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS All Total Score',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - Will',
               'column': 'pal_will',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Will', 'section': 'ADVANCE CARE PLANS/WISHES'}]},
              {'label': 'PAL - IPOS Q2 Other Symptoms 2',
               'column': 'pal_ipos_q2_other_symptoms_2',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2 Other Symptoms 2',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q2 Other Symptoms 3',
               'column': 'pal_ipos_q2_other_symptoms_3',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q2 Other Symptoms 3',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - IPOS Q1 Comment 1',
               'column': 'pal_ipos_q1_comment_1',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q1 Comment 1',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - Discharge Reason',
               'column': 'pal_discharge_reason',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Discharge Reason', 'section': 'ACTIONS/OUTCOMES'}]},
              {'label': 'PAL - IPOS Q1 Comment 2',
               'column': 'pal_ipos_q1_comment_2',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - IPOS Q1 Comment 2',
                            'section': 'PERFORMANCE STATUS - IPOS'}]},
              {'label': 'PAL - Seen by 1 Contact Details',
               'column': 'pal_seen_by_1_contact_details',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PAL - Seen by 1 Contact Details',
                            'section': 'CONTACT DETAILS'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'maternity_vte_risk_assessment',
 'display': 'Maternity and obstetric VTE risk assessment',
 'source_displays': ['Maternity VTE Risk Assessment', 'Obstetrics VTE Risk Assessment'],
 'carry_pregnancy_link': True,
 'elements': [{'label': 'Age>35 / Parity >3',
               'column': 'age_35_parity_3',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Age>35 / Parity >3',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Ob VTE Obesity',
               'column': 'ob_vte_obesity',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Ob VTE Obesity',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Obstetric VTE Risk Assessment Type',
               'column': 'obstetric_vte_risk_assessment_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Obstetric VTE Risk Assessment Type',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Obstetric VTE Risk Total v2',
               'column': 'obstetric_vte_risk_total_v2',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Obstetric VTE Risk Total v2',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Previous VTE',
               'column': 'previous_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Previous VTE', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Smoker',
               'column': 'smoker',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Smoker', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Current Systemic Infection',
               'column': 'current_systemic_infection',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Current Systemic Infection',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Dehydration/Reduced Immobility/ART/IVF',
               'column': 'dehydration_reduced_immobility_art_ivf',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Dehydration/Reduced Immobility/ART/IVF',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Family History of VTE',
               'column': 'family_history_of_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Family History of VTE',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Gross Varicose Veins',
               'column': 'gross_varicose_veins',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Gross Varicose Veins',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Hyperemesis',
               'column': 'hyperemesis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Hyperemesis', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Medical Comorbidities',
               'column': 'medical_comorbidities',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Medical Comorbidities',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'OHSS (Overian Hyperstimulation Syndrome)',
               'column': 'ohss_overian_hyperstimulation_syndrome',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'OHSS (Overian Hyperstimulation Syndrome)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Pre-eclampsia',
               'column': 'pre_eclampsia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pre-eclampsia', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Surg procedure in this preg or <=6 weeks',
               'column': 'surg_procedure_in_this_preg_or_6_weeks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Surg procedure in this preg or <=6 weeks',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'VTE Known Thrombophilia',
               'column': 'vte_known_thrombophilia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'VTE Known Thrombophilia',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Patient at Risk of VTE',
               'column': 'patient_at_risk_of_vte',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient at Risk of VTE',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Person Completing Form (VTE)',
               'column': 'person_completing_form_vte',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Person Completing Form (VTE)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Height/Length Measured',
               'column': 'height_length_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Height/Length Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Weight Measured',
               'column': 'weight_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Weight Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': '(System Use)VTE Maternity Assessment Status',
               'column': 'system_use_vte_maternity_assessment_status',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': '(System Use)VTE Maternity Assessment Status',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Maternity VTE total risk score',
               'column': 'maternity_vte_total_risk_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE total risk score',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Maternity VTE transient risk score',
               'column': 'maternity_vte_transient_risk_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE transient risk score',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Maternity VTE Low Trans risk',
               'column': 'maternity_vte_low_trans_risk',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Maternity VTE Low Trans risk', 'section': 'MATERNITY VTE'}]},
              {'label': 'Caesarean Section in Labour',
               'column': 'caesarean_section_in_labour',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Caesarean Section in Labour',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Elective Caesarean Section',
               'column': 'elective_caesarean_section',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Elective Caesarean Section',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Mid-Cavity or Rotational Forceps',
               'column': 'mid_cavity_or_rotational_forceps',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Mid-Cavity or Rotational Forceps',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'PPH (1 litre or more and/or transfusion)',
               'column': 'pph_1_litre_or_more_and_or_transfusion',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PPH (1 litre or more and/or transfusion)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Preterm birth this pregnancy',
               'column': 'preterm_birth_this_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Preterm birth this pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Prolonged Labour (over 24 hours)',
               'column': 'prolonged_labour_over_24_hours',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Prolonged Labour (over 24 hours)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Still birth this pregnancy',
               'column': 'still_birth_this_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Still birth this pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Multiple Pregnancy',
               'column': 'multiple_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Multiple Pregnancy',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Telephone Number',
               'column': 'telephone_number',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Telephone Number',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Maternity VTE action plan',
               'column': 'maternity_vte_action_plan',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE action plan', 'section': 'MATERNITY VTE'}]},
              {'label': 'Body Mass Index Measured',
               'column': 'body_mass_index_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Body Mass Index Measured',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'BMI',
               'column': 'bmi',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'BMI', 'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Multiple Pregnancy (Twins or more)',
               'column': 'multiple_pregnancy_twins_or_more',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Multiple Pregnancy (Twins or more)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Maternity VTE permanent risk score',
               'column': 'maternity_vte_permanent_risk_score',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE permanent risk score',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Appropriate Thrombo for Dr to prescribe',
               'column': 'appropriate_thrombo_for_dr_to_prescribe',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Appropriate Thrombo for Dr to prescribe',
                            'section': 'VTE TREATMENT PLAN'}]},
              {'label': 'Maternity VTE Low Perm risk',
               'column': 'maternity_vte_low_perm_risk',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Maternity VTE Low Perm risk', 'section': 'MATERNITY VTE'}]},
              {'label': 'Contraindication to LMWH or Heparin',
               'column': 'contraindication_to_lmwh_or_heparin',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Contraindication to LMWH or Heparin',
                            'section': 'VTE TREATMENT PLAN'}]},
              {'label': 'Heparin Type',
               'column': 'heparin_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Heparin Type', 'section': 'VTE TREATMENT PLAN'}]},
              {'label': 'Maternity VTE Intermediate Trans risk',
               'column': 'maternity_vte_intermediate_trans_risk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE Intermediate Trans risk',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Medical Comorbidities Type',
               'column': 'medical_comorbidities_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Medical Comorbidities Type',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'Maternity VTE Intermediate Perm risk',
               'column': 'maternity_vte_intermediate_perm_risk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Maternity VTE Intermediate Perm risk',
                            'section': 'MATERNITY VTE'}]},
              {'label': 'Antiphospholipid antibodies',
               'column': 'antiphospholipid_antibodies',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Antiphospholipid antibodies',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'FactorV leiden Heterozygous',
               'column': 'factorv_leiden_heterozygous',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'FactorV leiden Heterozygous',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]},
              {'label': 'ProThrombin Gene Mutation (heterozygous)',
               'column': 'prothrombin_gene_mutation_heterozygous',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'ProThrombin Gene Mutation (heterozygous)',
                            'section': 'OBSTETRIC VTE RISK ASSESSMENT'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'maternity_personalised_care_plan',
 'display': 'Maternity personalised care plan',
 'source_displays': ['Maternity Personalised Care Plan (PCP)'],
 'carry_pregnancy_link': True,
 'elements': [{'label': 'Named Midwife',
               'column': 'named_midwife',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Named Midwife', 'section': 'PCP TEAM'}]},
              {'label': 'Midwifery Team',
               'column': 'midwifery_team',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Midwifery Team', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Current Gestation Postnatal',
               'column': 'pcp_current_gestation_postnatal',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation Postnatal', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Current Gestation 28wks',
               'column': 'pcp_current_gestation_28wks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation 28wks', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Current Gestation 16wks',
               'column': 'pcp_current_gestation_16wks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation 16wks', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Gestation Weeks',
               'column': 'pcp_gestation_weeks',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Gestation Weeks', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'PCP Gestation Days',
               'column': 'pcp_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Gestation Days', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'Midwife at Discussion',
               'column': 'midwife_at_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Midwife at Discussion', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'Date',
               'column': 'date_16_27_6_weeks',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'Date', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'PCP Intended Place of Birth',
               'column': 'pcp_intended_place_of_birth',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Intended Place of Birth', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'PCP Personalised care plan discussed',
               'column': 'pcp_personalised_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Personalised care plan discussed',
                            'section': '16 - 27+6 WEEKS'}]},
              {'label': 'PCP Named Midwife Changed?',
               'column': 'pcp_named_midwife_changed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Named Midwife Changed?', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Midwifery Team  Changed?',
               'column': 'pcp_midwifery_team_changed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Midwifery Team  Changed?', 'section': 'PCP TEAM'}]},
              {'label': 'PCP 16WK Comments',
               'column': 'pcp_16wk_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 16WK Comments', 'section': '16 - 27+6 WEEKS'}]},
              {'label': 'PCP 34-37 Date',
               'column': 'pcp_34_37_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Date', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 34-37 Gestation WK',
               'column': 'pcp_34_37_gestation_wk',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Gestation WK', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 34-37 Midwife Discussion',
               'column': 'pcp_34_37_midwife_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Midwife Discussion', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 34-37 Gestation Days',
               'column': 'pcp_34_37_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Gestation Days', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP Current Gestation 34wks',
               'column': 'pcp_current_gestation_34wks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation 34wks', 'section': 'PCP TEAM'}]},
              {'label': 'PCP 34-37 Care Plan Discussed',
               'column': 'pcp_34_37_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Care Plan Discussed', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 34-37 Birth Plan Completed',
               'column': 'pcp_34_37_birth_plan_completed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Birth Plan Completed', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 34-37 Intended Place of Birth',
               'column': 'pcp_34_37_intended_place_of_birth',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34-37 Intended Place of Birth',
                            'section': '34-37 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Date',
               'column': 'pcp_28_33_6_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Date', 'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Gestation Wk',
               'column': 'pcp_28_33_6_gestation_wk',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Gestation Wk', 'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Gestation Days',
               'column': 'pcp_28_33_6_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Gestation Days',
                            'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Pt booked CPP',
               'column': 'pcp_28_33_6_pt_booked_cpp',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Pt booked CPP', 'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Care Plan Discussed',
               'column': 'pcp_28_33_6_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Care Plan Discussed',
                            'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 28 - 33+6 Midwife at Discussion',
               'column': 'pcp_28_33_6_midwife_at_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 33+6 Midwife at Discussion',
                            'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP 34 - 37 WK Comments',
               'column': 'pcp_34_37_wk_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 34 - 37 WK Comments', 'section': '34-37 WEEKS'}]},
              {'label': 'PCP 28 WK Comments',
               'column': 'pcp_28_wk_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 WK Comments', 'section': '28 - 33+6 WEEKS'}]},
              {'label': 'PCP Name of midwife visit 3',
               'column': 'pcp_name_of_midwife_visit_3',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Name of midwife visit 3',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 3'}]},
              {'label': 'PCP Visit 3 Postnatal day',
               'column': 'pcp_visit_3_postnatal_day',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 3 Postnatal day',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 3'}]},
              {'label': 'PCP Visit 3 Date',
               'column': 'pcp_visit_3_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 3 Date',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 3'}]},
              {'label': 'Postnatal (Community) Visit 3',
               'column': 'postnatal_community_visit_3',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Postnatal (Community) Visit 3', 'section': 'PCP TEAM'}]},
              {'label': 'PCP P Natal Visit 3 Comments',
               'column': 'pcp_p_natal_visit_3_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP P Natal Visit 3 Comments',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 3'}]},
              {'label': 'Buddy Midwife 1',
               'column': 'buddy_midwife_1',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Buddy Midwife 1', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Name of midwife visit 1',
               'column': 'pcp_name_of_midwife_visit_1',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Name of midwife visit 1',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 1'}]},
              {'label': 'PCP Visit 1 Postnatal day',
               'column': 'pcp_visit_1_postnatal_day',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 1 Postnatal day',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 1'}]},
              {'label': 'Postnatal (Community) Visit 1',
               'column': 'postnatal_community_visit_1',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Postnatal (Community) Visit 1', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Visit 1 Date',
               'column': 'pcp_visit_1_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 1 Date',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 1'}]},
              {'label': 'PCP P Natal Visit 1 Comments',
               'column': 'pcp_p_natal_visit_1_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP P Natal Visit 1 Comments',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 1'}]},
              {'label': 'PCP 28 - 34+6 Gestation Wk',
               'column': 'pcp_28_34_6_gestation_wk',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Gestation Wk', 'section': '28 - 34+6 WEEKS'}]},
              {'label': 'PCP 28 - 34+6 Midwife at Discussion',
               'column': 'pcp_28_34_6_midwife_at_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Midwife at Discussion',
                            'section': '28 - 34+6 WEEKS'}]},
              {'label': 'PCP 28 - 34+6 Gestation Days',
               'column': 'pcp_28_34_6_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Gestation Days',
                            'section': '28 - 34+6 WEEKS'}]},
              {'label': 'PCP 28 - 34+6 Date',
               'column': 'pcp_28_34_6_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Date', 'section': '28 - 34+6 WEEKS'}]},
              {'label': '28-34+6 weeks Visit',
               'column': 'item_28_34_6_weeks_visit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': '28-34+6 weeks Visit', 'section': 'PCP TEAM'}]},
              {'label': 'PCP 28 - 34+6  Pt booked CPP',
               'column': 'pcp_28_34_6_pt_booked_cpp',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6  Pt booked CPP',
                            'section': '28 - 34+6 WEEKS'}]},
              {'label': 'PCP 28 - 34+6 Care Plan Discussed',
               'column': 'pcp_28_34_6_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Care Plan Discussed',
                            'section': '28 - 34+6 WEEKS'}]},
              {'label': 'PCP 28 - 34+6 Comments',
               'column': 'pcp_28_34_6_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 28 - 34+6 Comments', 'section': '28 - 34+6 WEEKS'}]},
              {'label': '16-27 +6weeks Visit',
               'column': 'item_16_27_6weeks_visit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': '16-27 +6weeks Visit', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Name of midwife visit 2',
               'column': 'pcp_name_of_midwife_visit_2',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Name of midwife visit 2',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 2'}]},
              {'label': 'PCP Visit 2 Postnatal day',
               'column': 'pcp_visit_2_postnatal_day',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 2 Postnatal day',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 2'}]},
              {'label': 'PCP Visit 2 Date',
               'column': 'pcp_visit_2_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Visit 2 Date',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 2'}]},
              {'label': 'Postnatal (Community) Visit 2',
               'column': 'postnatal_community_visit_2',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Postnatal (Community) Visit 2', 'section': 'PCP TEAM'}]},
              {'label': 'PCP P Natal Visit 2 Comments',
               'column': 'pcp_p_natal_visit_2_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP P Natal Visit 2 Comments',
                            'section': 'POSTNATAL (COMMUNITY) VISIT 2'}]},
              {'label': 'PCP Buddy Midwife 2 Changed?',
               'column': 'pcp_buddy_midwife_2_changed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Buddy Midwife 2 Changed?', 'section': 'PCP TEAM'}]},
              {'label': 'PCP Current Gestation 35wks',
               'column': 'pcp_current_gestation_35wks',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation 35wks', 'section': 'PCP TEAM'}]},
              {'label': 'PCP 35-36+6 Gestation Days',
               'column': 'pcp_35_36_6_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Gestation Days', 'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Gestation WK',
               'column': 'pcp_35_36_6_gestation_wk',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Gestation WK', 'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Midwife Discussion',
               'column': 'pcp_35_36_6_midwife_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Midwife Discussion',
                            'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Date',
               'column': 'pcp_35_36_6_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Date', 'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Birth Plan Completed',
               'column': 'pcp_35_36_6_birth_plan_completed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Birth Plan Completed',
                            'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Care Plan Discussed',
               'column': 'pcp_35_36_6_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Care Plan Discussed',
                            'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP 35-36+6 Intended Place of Birth',
               'column': 'pcp_35_36_6_intended_place_of_birth',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 Intended Place of Birth',
                            'section': '35-36+6 WEEKS'}]},
              {'label': 'PCP Current Gestation greater than 37',
               'column': 'pcp_current_gestation_greater_than_37',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Current Gestation greater than 37',
                            'section': 'PCP TEAM'}]},
              {'label': 'PCP Greater than 37 Gestation Days',
               'column': 'pcp_greater_than_37_gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Gestation Days',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Gestation Weeks',
               'column': 'pcp_greater_than_37_gestation_weeks',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Gestation Weeks',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Midwife Discussion',
               'column': 'pcp_greater_than_37_midwife_discussion',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Midwife Discussion',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Date',
               'column': 'pcp_greater_than_37_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Date',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'Greater than 37 weeks Visit',
               'column': 'greater_than_37_weeks_visit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Greater than 37 weeks Visit', 'section': 'PCP TEAM'}]},
              {'label': 'Any new risk indentified after 36 wks appointment',
               'column': 'any_new_risk_indentified_after_36_wks_appointment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any new risk indentified after 36 wks appointment',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Birth Plan Completed?',
               'column': 'pcp_greater_than_37_birth_plan_completed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Birth Plan Completed?',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Care Plan Discussed',
               'column': 'pcp_greater_than_37_care_plan_discussed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Care Plan Discussed',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Intended Place of Birth',
               'column': 'pcp_greater_than_37_intended_place_of_birth',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Intended Place of Birth',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP Greater than 37 Comments',
               'column': 'pcp_greater_than_37_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Greater than 37 Comments',
                            'section': 'GREATER THAN 37 WEEKS'}]},
              {'label': 'PCP 35-36+6 WK Comments',
               'column': 'pcp_35_36_6_wk_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP 35-36+6 WK Comments', 'section': '35-36+6 WEEKS'}]},
              {'label': 'Gestation days',
               'column': 'gestation_days',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Gestation days',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'Gestation weeks',
               'column': 'gestation_weeks',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Gestation weeks',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'Date',
               'column': 'date_maternity_pcp_maternity_carbon_monoxide_smoking',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'Date',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'Carbon Monoxide Reading Performed',
               'column': 'carbon_monoxide_reading_performed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Carbon Monoxide Reading Performed',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': '35-36+6 weeks Visit1',
               'column': 'item_35_36_6_weeks_visit1',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': '35-36+6 weeks Visit1', 'section': 'PCP TEAM'}]},
              {'label': 'MCO Smoking Status Mother',
               'column': 'mco_smoking_status_mother',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'MCO Smoking Status Mother',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'PCP Carbon Monoxide',
               'column': 'pcp_carbon_monoxide',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Carbon Monoxide',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'PCP Named Buddy 1 Changed?',
               'column': 'pcp_named_buddy_1_changed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Named Buddy 1 Changed?', 'section': 'PCP TEAM'}]},
              {'label': 'Carbon Monoxide Reading',
               'column': 'carbon_monoxide_reading',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Carbon Monoxide Reading',
                            'section': 'MATERNITY PCP MATERNITY CARBON MONOXIDE / SMOKING'}]},
              {'label': 'PCP Reason for change',
               'column': 'pcp_reason_for_change',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'PCP Reason for change', 'section': 'PCP TEAM'}]},
              {'label': 'Buddy Midwife 2',
               'column': 'buddy_midwife_2',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Buddy Midwife 2', 'section': 'PCP TEAM'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'family_origin_questionnaire',
 'display': 'Family origin questionnaire',
 'source_displays': ['Family Origin Questionaire'],
 'carry_pregnancy_link': True,
 'elements': [{'label': 'H Is this pregnancy a result of EGG DONATION',
               'column': 'h_is_this_pregnancy_a_result_of_egg_donation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'H Is this pregnancy a result of EGG DONATION',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'H Unknown paternal family origins',
               'column': 'h_unknown_paternal_family_origins',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'H Unknown paternal family origins',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'H Has the mother had a bone marrow transplant',
               'column': 'h_has_the_mother_had_a_bone_marrow_transplant',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'H Has the mother had a bone marrow transplant',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'H Is the pregnancy a result of SPERM DONATION?',
               'column': 'h_is_the_pregnancy_a_result_of_sperm_donation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'H Is the pregnancy a result of SPERM DONATION?',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'H Unknown maternal family origins',
               'column': 'h_unknown_maternal_family_origins',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'H Unknown maternal family origins',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'family Origin M',
               'column': 'family_origin_m',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'family Origin M',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'Family Origin F',
               'column': 'family_origin_f',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Family Origin F',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'Gestation At Time of Test',
               'column': 'gestation_at_time_of_test',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Gestation At Time of Test',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]},
              {'label': 'FOQ Additional Details',
               'column': 'foq_additional_details',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'FOQ Additional Details',
                            'section': 'FAMILY ORIGIN QUESTIONAIRE NEW V1'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'preoperative_assessment',
 'display': 'Pre-operative assessment (both form eras)',
 'source_displays': ['Pre Op Assessment', 'Preoperative Assessment - MODEL'],
 'elements': [{'label': 'Planned Procedure',
               'column': 'planned_procedure',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Planned Procedure',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'},
                           {'label': 'Planned Procedure', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'POA Has Allergies',
               'column': 'poa_has_allergies',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Has Allergies',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Language Spoken',
               'column': 'language_spoken',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Language Spoken',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'},
                           {'label': 'Language Spoken', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'POA Hypertension',
               'column': 'poa_hypertension',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Hypertension', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Anaesthetic History',
               'column': 'poa_anaesthetic_history',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Anaesthetic History', 'section': 'ANAESTHETIC HISTORY'}]},
              {'label': 'POA PDR Assessment Req',
               'column': 'poa_pdr_assessment_req',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA PDR Assessment Req',
                            'section': 'PRION DISEASE/VARIENT CJD'}]},
              {'label': 'POA Palpitations/Irregular Heartbeat',
               'column': 'poa_palpitations_irregular_heartbeat',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Palpitations/Irregular Heartbeat',
                            'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Cardiac Surgery',
               'column': 'poa_cardiac_surgery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Cardiac Surgery', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA MI',
               'column': 'poa_mi',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA MI', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'Relative anaesthetic problem',
               'column': 'relative_anaesthetic_problem',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Relative anaesthetic problem',
                            'section': 'ANAESTHETIC HISTORY'}]},
              {'label': 'POA Heart Failure/Cardiomyopathy',
               'column': 'poa_heart_failure_cardiomyopathy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Heart Failure/Cardiomyopathy',
                            'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Chest Pain/Angina',
               'column': 'poa_chest_pain_angina',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Chest Pain/Angina', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Valve Problem',
               'column': 'poa_valve_problem',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Valve Problem', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA DVT',
               'column': 'poa_dvt',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA DVT', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA PE',
               'column': 'poa_pe',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA PE', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Fainting/Syncope',
               'column': 'poa_fainting_syncope',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Fainting/Syncope', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Murmur/Rheumatic Fever',
               'column': 'poa_murmur_rheumatic_fever',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Murmur/Rheumatic Fever',
                            'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Ankle Oedema',
               'column': 'poa_ankle_oedema',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Ankle Oedema', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA SOB',
               'column': 'poa_sob',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA SOB', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Transfusion',
               'column': 'poa_transfusion',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Transfusion', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Endocrine/Metabolic',
               'column': 'poa_endocrine_metabolic',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Endocrine/Metabolic', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Gender',
               'column': 'poa_gender_pre_operative_assessment_screening_i',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Gender',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Able to Communicate',
               'column': 'poa_able_to_communicate',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Able to Communicate',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'},
                           {'label': 'POA Able to Communicate', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'POA Haematology',
               'column': 'poa_haematology',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Haematology', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Psychiatric Problems',
               'column': 'poa_psychiatric_problems',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Psychiatric Problems', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Gastrointestinal',
               'column': 'poa_gastrointestinal',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Gastrointestinal', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Renal Disease',
               'column': 'poa_renal_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Renal Disease', 'section': 'OTHER SYSTEMS'},
                           {'label': 'POA Renal Disease', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'POA Case Type',
               'column': 'poa_case_type',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Case Type',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Neurological',
               'column': 'poa_neurological',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Neurological', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Anaesthesia Proposed',
               'column': 'poa_anaesthesia_proposed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Anaesthesia Proposed',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Musculoskeletal',
               'column': 'poa_musculoskeletal',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Musculoskeletal', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'Swabs Required',
               'column': 'swabs_required',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Swabs Required',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'Height/Length Measured',
               'column': 'height_length_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Height/Length Measured', 'section': 'PHYSICAL ASSESSMENT'},
                           {'label': 'Height/Length Measured',
                            'section': 'HEIGHT/WEIGHT/ALLERGY'}]},
              {'label': 'Weight Measured',
               'column': 'weight_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Weight Measured', 'section': 'PHYSICAL ASSESSMENT'},
                           {'label': 'Weight Measured', 'section': 'HEIGHT/WEIGHT/ALLERGY'}]},
              {'label': 'POA Respiratory',
               'column': 'poa_respiratory',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Respiratory', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA TCI Date',
               'column': 'poa_tci_date',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA TCI Date',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Next of Kin',
               'column': 'poa_next_of_kin',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Next of Kin',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Pregnancy',
               'column': 'poa_pregnancy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Pregnancy', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'Systolic Blood Pressure',
               'column': 'systolic_blood_pressure',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Systolic Blood Pressure', 'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'Peripheral Pulse Rate',
               'column': 'peripheral_pulse_rate',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Peripheral Pulse Rate', 'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'SpO2',
               'column': 'spo2',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'SpO2', 'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'Diastolic Blood Pressure',
               'column': 'diastolic_blood_pressure',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Diastolic Blood Pressure',
                            'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'Mean Arterial Pressure, Cuff',
               'column': 'mean_arterial_pressure_cuff',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Mean Arterial Pressure, Cuff',
                            'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'POA Dependant Children',
               'column': 'poa_dependant_children',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Dependant Children', 'section': 'SAFEGUARDING'}]},
              {'label': 'POA Carer',
               'column': 'poa_carer',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Carer', 'section': 'SAFEGUARDING'}]},
              {'label': 'POA Age',
               'column': 'poa_age',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POA Age',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Breathing Problems',
               'column': 'poa_breathing_problems',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Breathing Problems', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA Learning Disabilities',
               'column': 'poa_learning_disabilities',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Learning Disabilities', 'section': 'SAFEGUARDING'}]},
              {'label': 'POA Patient at Risk',
               'column': 'poa_patient_at_risk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Patient at Risk', 'section': 'SAFEGUARDING'}]},
              {'label': 'POA Assisted By',
               'column': 'poa_assisted_by',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Assisted By', 'section': 'SOCIAL ASSESSMENT'}]},
              {'label': 'POA Mental Health',
               'column': 'poa_mental_health',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Mental Health', 'section': 'SAFEGUARDING'}]},
              {'label': 'POA Coping on Discharge',
               'column': 'poa_coping_on_discharge',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Coping on Discharge', 'section': 'SOCIAL ASSESSMENT'}]},
              {'label': 'Responsible adult present to care for patient',
               'column': 'responsible_adult_present_to_care_for_patient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Responsible adult present to care for patient',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Easy access to bathroom/toilet',
               'column': 'easy_access_to_bathroom_toilet',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Easy access to bathroom/toilet',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Dependant Relatives',
               'column': 'poa_dependant_relatives',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Dependant Relatives', 'section': 'SOCIAL ASSESSMENT'}]},
              {'label': 'Stairs to climb at home',
               'column': 'stairs_to_climb_at_home',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Stairs to climb at home',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Telephone access',
               'column': 'telephone_access',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Telephone access',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA How Far Can Patient Walk',
               'column': 'poa_how_far_can_patient_walk',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'POA How Far Can Patient Walk',
                            'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Alcohol',
               'column': 'poa_alcohol',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POA Alcohol', 'section': 'OTHER SYSTEMS'}]},
              {'label': 'Date Swabs Taken',
               'column': 'date_swabs_taken',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'Date Swabs Taken',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Good Mouth Opening',
               'column': 'poa_good_mouth_opening',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Good Mouth Opening', 'section': 'AIRWAY'}]},
              {'label': 'POA Available for Surgery',
               'column': 'poa_available_for_surgery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Available for Surgery', 'section': 'SOCIAL ASSESSMENT'}]},
              {'label': 'POA Consultant',
               'column': 'poa_consultant',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POA Consultant',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Uses Herbal Medicine',
               'column': 'uses_herbal_medicine',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Uses Herbal Medicine',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Good C-Spine Movement',
               'column': 'poa_good_c_spine_movement',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Good C-Spine Movement', 'section': 'AIRWAY'}]},
              {'label': 'POA Safe Environment',
               'column': 'poa_safe_environment',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Safe Environment', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'POA Communication',
               'column': 'poa_communication',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Communication', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'Travel Distance < 1Hr from Hospital',
               'column': 'travel_distance_1hr_from_hospital',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Travel Distance < 1Hr from Hospital',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Have you ever had MRSA',
               'column': 'have_you_ever_had_mrsa',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Have you ever had MRSA',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Problems with Heart',
               'column': 'poa_problems_with_heart',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Problems with Heart', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA Elimination',
               'column': 'poa_elimination',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Elimination', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'POA Mobilising',
               'column': 'poa_mobilising',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Mobilising', 'section': 'NURSE ASSESSMENT II'}]},
              {'label': 'POA Personal Care',
               'column': 'poa_personal_care',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Personal Care', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'POA Eat and Drink',
               'column': 'poa_eat_and_drink',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Eat and Drink', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'Uses Recreational Drugs',
               'column': 'uses_recreational_drugs',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Uses Recreational Drugs',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Breathing',
               'column': 'poa_breathing',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Breathing', 'section': 'NURSE ASSESSMENT I'}]},
              {'label': 'POA Date of Assessment',
               'column': 'poa_date_of_assessment',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'POA Date of Assessment',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'Drug Medication Name',
               'column': 'drug_medication_name',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Drug Medication Name',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Mallampati',
               'column': 'poa_mallampati',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Mallampati', 'section': 'AIRWAY'}]},
              {'label': 'POA Blood transfusion in last 3 months',
               'column': 'poa_blood_transfusion_in_last_3_months',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Blood transfusion in last 3 months',
                            'section': 'OTHER SYSTEMS'}]},
              {'label': 'POA Unavailable for Surgery',
               'column': 'poa_unavailable_for_surgery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Unavailable for Surgery',
                            'section': 'SOCIAL ASSESSMENT'}]},
              {'label': 'POA Work/Leisure',
               'column': 'poa_work_leisure',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Work/Leisure', 'section': 'NURSE ASSESSMENT II'}]},
              {'label': 'POA Anaesthetic Problems',
               'column': 'poa_anaesthetic_problems',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Anaesthetic Problems',
                            'section': 'ANAESTHETIC HISTORY'}]},
              {'label': 'Worries, hopes and concerns',
               'column': 'worries_hopes_and_concerns',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Worries, hopes and concerns',
                            'section': 'NURSE ASSESSMENT II'}]},
              {'label': 'Respiratory Rate',
               'column': 'respiratory_rate',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Respiratory Rate', 'section': 'PHYSICAL ASSESSMENT'}]},
              {'label': 'Dentition Freetext',
               'column': 'dentition_freetext',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Dentition Freetext', 'section': 'AIRWAY'}]},
              {'label': 'POA Pacemaker/IED',
               'column': 'poa_pacemaker_ied',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Pacemaker/IED', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'Drug Dose',
               'column': 'drug_dose',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Drug Dose',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Primary Consent',
               'column': 'poa_primary_consent',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Primary Consent',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING I'}]},
              {'label': 'POA Body Temperature',
               'column': 'poa_body_temperature',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Body Temperature', 'section': 'NURSE ASSESSMENT II'}]},
              {'label': 'POA Cough',
               'column': 'poa_cough',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Cough', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA Sputum',
               'column': 'poa_sputum',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Sputum', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA CPAP',
               'column': 'poa_cpap',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA CPAP', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA Wheeze',
               'column': 'poa_wheeze',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Wheeze', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA Home Oxygen',
               'column': 'poa_home_oxygen',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Home Oxygen', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA Hospital Admissions',
               'column': 'poa_hospital_admissions',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Hospital Admissions',
                            'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'POA ICU Admissions',
               'column': 'poa_icu_admissions',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA ICU Admissions', 'section': 'CARDIORESPIRATORY II'}]},
              {'label': 'Drug Frequency',
               'column': 'drug_frequency',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Drug Frequency',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'Drug Route',
               'column': 'drug_route',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Drug Route',
                            'section': 'PRE OPERATIVE ASSESSMENT SCREENING LL'}]},
              {'label': 'POA Cause to Stop',
               'column': 'poa_cause_to_stop',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Cause to Stop', 'section': 'CARDIORESPIRATORY I'}]},
              {'label': 'POA NURSE SIGNATURE',
               'column': 'poa_nurse_signature',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'POA NURSE SIGNATURE', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'Adult or Paediatric',
               'column': 'adult_or_paediatric',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Adult or Paediatric', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'POA Gynae Dummy',
               'column': 'poa_gynae_dummy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Gynae Dummy', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'POA Nurse Signature Date and Time',
               'column': 'poa_nurse_signature_date_and_time',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'POA Nurse Signature Date and Time',
                            'section': 'PATIENT INFORMATION'}]},
              {'label': 'Fit for Surgery',
               'column': 'fit_for_surgery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Fit for Surgery', 'section': 'INITIAL OUTCOME'}]},
              {'label': 'Had General Anaesthetic Previously',
               'column': 'had_general_anaesthetic_previously',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Had General Anaesthetic Previously',
                            'section': 'ANAESTHETIC/SURGICAL HISTORY'}]},
              {'label': 'Relatives History of Anaes Reaction',
               'column': 'relatives_history_of_anaes_reaction',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Relatives History of Anaes Reaction',
                            'section': 'ANAESTHETIC/SURGICAL HISTORY'}]},
              {'label': 'Recreational drugs',
               'column': 'recreational_drugs',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Recreational drugs', 'section': 'MEDICATION HISTORY'}]},
              {'label': 'Objects to Blood Transfusions',
               'column': 'objects_to_blood_transfusions',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Objects to Blood Transfusions',
                            'section': 'HAEMATOLOGICAL REVIEW'}]},
              {'label': 'Limited Ability to Walk or Climb Stairs',
               'column': 'limited_ability_to_walk_or_climb_stairs',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Limited Ability to Walk or Climb Stairs',
                            'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Patient Walking Distance',
               'column': 'patient_walking_distance',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Patient Walking Distance',
                            'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Stairs at Home',
               'column': 'stairs_at_home',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Stairs at Home', 'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'MRSA Decolonisation Protocol Started',
               'column': 'mrsa_decolonisation_protocol_started',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'MRSA Decolonisation Protocol Started',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Patient has had MRSA',
               'column': 'patient_has_had_mrsa',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient has had MRSA',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Has Had Brain or Spinal Cord Surgery',
               'column': 'has_had_brain_or_spinal_cord_surgery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Has Had Brain or Spinal Cord Surgery',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Patient has had CRO',
               'column': 'patient_has_had_cro',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient has had CRO',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Has Had Growth Hormone or Gonadotrophin',
               'column': 'has_had_growth_hormone_or_gonadotrophin',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Has Had Growth Hormone or Gonadotrophin',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Patient has had VRE',
               'column': 'patient_has_had_vre',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Patient has had VRE',
                            'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'DASI Results',
               'column': 'dasi_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'DASI Results', 'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Haematological Results',
               'column': 'haematological_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Haematological Results',
                            'section': 'HAEMATOLOGICAL REVIEW'}]},
              {'label': 'Respiratory Investigation Orders',
               'column': 'respiratory_investigation_orders',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Respiratory Investigation Orders',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Cardiac Orders',
               'column': 'cardiac_orders',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Cardiac Orders', 'section': 'CARDIAC REVIEW'}]},
              {'label': 'Cardiac Results',
               'column': 'cardiac_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Cardiac Results', 'section': 'CARDIAC REVIEW'}]},
              {'label': 'Abdominal Exam Results',
               'column': 'abdominal_exam_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Abdominal Exam Results', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Moca Score',
               'column': 'moca_score',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Moca Score', 'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'Endocrine Results',
               'column': 'endocrine_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Endocrine Results', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Any Known Cardiac Problems',
               'column': 'any_known_cardiac_problems',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any Known Cardiac Problems', 'section': 'CARDIAC REVIEW'}]},
              {'label': 'Gender = Male',
               'column': 'gender_male',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Gender = Male', 'section': 'SLEEP APNOEA REVIEW (ADULT)'}]},
              {'label': 'Age > 50',
               'column': 'age_50',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Age > 50', 'section': 'SLEEP APNOEA REVIEW (ADULT)'}]},
              {'label': 'Diagnosis of OSA',
               'column': 'diagnosis_of_osa',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Diagnosis of OSA',
                            'section': 'SLEEP APNOEA REVIEW (ADULT)'}]},
              {'label': 'Clinical Frailty Scale',
               'column': 'clinical_frailty_scale',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Clinical Frailty Scale',
                            'section': 'CLINICAL FRAILTY SCALE'}]},
              {'label': 'POA Diabetes',
               'column': 'poa_diabetes',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Diabetes', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Able to Lie Flat',
               'column': 'able_to_lie_flat',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Able to Lie Flat', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Appointment Type',
               'column': 'appointment_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Appointment Type', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'Gastro oesphageal reflux',
               'column': 'gastro_oesphageal_reflux',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Gastro oesphageal reflux', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Liver disease',
               'column': 'liver_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Liver disease', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Heartburn',
               'column': 'heartburn',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Heartburn', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Hiatus Hernia',
               'column': 'hiatus_hernia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Hiatus Hernia', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Hypothyroidism',
               'column': 'hypothyroidism',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Hypothyroidism', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Inflammatory bowel disease',
               'column': 'inflammatory_bowel_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Inflammatory bowel disease', 'section': 'ABDOMINAL REVIEW'}]},
              {'label': 'Hyperthyroidism',
               'column': 'hyperthyroidism',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Hyperthyroidism', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Addisons',
               'column': 'addisons',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Addisons', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Cushings',
               'column': 'cushings',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Cushings', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'POA Asthma',
               'column': 'poa_asthma',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Asthma', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Case Type',
               'column': 'case_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Case Type', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'Preoperative Information Provided',
               'column': 'preoperative_information_provided',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Preoperative Information Provided',
                            'section': 'INFORMATION DISCUSSED AND PROVIDED'}]},
              {'label': 'On Anticoagulants, Antiplatelets',
               'column': 'on_anticoagulants_antiplatelets',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'On Anticoagulants, Antiplatelets',
                            'section': 'MEDICATION HISTORY'}]},
              {'label': 'POA Anxiety',
               'column': 'poa_anxiety',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Anxiety',
                            'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'Has Obstructive Sleep Apnoea',
               'column': 'has_obstructive_sleep_apnoea',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Has Obstructive Sleep Apnoea',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'COPD, Chronic Lung Disease',
               'column': 'copd_chronic_lung_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'COPD, Chronic Lung Disease',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'POA Depression',
               'column': 'poa_depression',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Depression',
                            'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'Has Cystic Fibrosis',
               'column': 'has_cystic_fibrosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Has Cystic Fibrosis', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Recent Chest Infections',
               'column': 'recent_chest_infections',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Recent Chest Infections', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'On Home O2 or Nebulisers',
               'column': 'on_home_o2_or_nebulisers',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'On Home O2 or Nebulisers', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Has Had Tuberculosis',
               'column': 'has_had_tuberculosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Has Had Tuberculosis', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Recent Covid pneumonia',
               'column': 'recent_covid_pneumonia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Recent Covid pneumonia', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Pulmonary fibrosis',
               'column': 'pulmonary_fibrosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pulmonary fibrosis', 'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Daytime sleepiness, snoring, obs apnoea',
               'column': 'daytime_sleepiness_snoring_obs_apnoea',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Daytime sleepiness, snoring, obs apnoea',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Previous ITU Admission for Breathing',
               'column': 'previous_itu_admission_for_breathing',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Previous ITU Admission for Breathing',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Baseline Observations RT',
               'column': 'baseline_observations_rt',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Baseline Observations RT',
                            'section': 'BASELINE OBSERVATIONS'}]},
              {'label': 'POA Psychosis',
               'column': 'poa_psychosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Psychosis',
                            'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'Who do you live with',
               'column': 'who_do_you_live_with',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Who do you live with', 'section': 'SOCIAL HISTORY'}]},
              {'label': 'Any Known Neurological Problems',
               'column': 'any_known_neurological_problems',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any Known Neurological Problems',
                            'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'Interpreter Required',
               'column': 'interpreter_required',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Interpreter Required', 'section': 'PATIENT INFORMATION'}]},
              {'label': 'Bleomycin Administered in Past',
               'column': 'bleomycin_administered_in_past',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Bleomycin Administered in Past',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Any Pulmonary Embolus (PE) or DVT',
               'column': 'any_pulmonary_embolus_pe_or_dvt',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any Pulmonary Embolus (PE) or DVT',
                            'section': 'HAEMATOLOGICAL REVIEW'}]},
              {'label': 'Planned Procedure for POA Visit',
               'column': 'planned_procedure_for_poa_visit',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Planned Procedure for POA Visit',
                            'section': 'PATIENT INFORMATION'}]},
              {'label': 'Any Known Respiratory Problems',
               'column': 'any_known_respiratory_problems',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any Known Respiratory Problems',
                            'section': 'RESPIRATORY REVIEW'}]},
              {'label': 'Any Probs With Mobility, Muscle or Joint',
               'column': 'any_probs_with_mobility_muscle_or_joint',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Any Probs With Mobility, Muscle or Joint',
                            'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'POA - Shortness of breath',
               'column': 'poa_shortness_of_breath',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA - Shortness of breath', 'section': 'CARDIAC REVIEW'}]},
              {'label': 'Stroke or TIA',
               'column': 'stroke_or_tia',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Stroke or TIA', 'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'Parkinsons disease',
               'column': 'parkinsons_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Parkinsons disease', 'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'Multiple Sclerosis',
               'column': 'multiple_sclerosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Multiple Sclerosis', 'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'OPD Prescription Specialty Type',
               'column': 'opd_prescription_specialty_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'OPD Prescription Specialty Type',
                            'section': 'PATIENT INFORMATION'}]},
              {'label': 'Other',
               'column': 'other_surgical_infection_risk_assessment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Other', 'section': 'SURGICAL INFECTION RISK ASSESSMENT'}]},
              {'label': 'Epilepsy',
               'column': 'epilepsy',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Epilepsy', 'section': 'NEUROLOGICAL REVIEW'}]},
              {'label': 'Herbal/compimentary medicine',
               'column': 'herbal_compimentary_medicine',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Herbal/compimentary medicine',
                            'section': 'MEDICATION HISTORY'}]},
              {'label': 'Body Mass Index Measured',
               'column': 'body_mass_index_measured',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Body Mass Index Measured',
                            'section': 'HEIGHT/WEIGHT/ALLERGY'}]},
              {'label': 'Osteoarthritis',
               'column': 'osteoarthritis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Osteoarthritis', 'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Neck Movement',
               'column': 'neck_movement',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Neck Movement', 'section': 'AIRWAY REVIEW'}]},
              {'label': 'POA Other Endocrine',
               'column': 'poa_other_endocrine_behavioural_psychological_review',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'POA Other Endocrine',
                            'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'Osteoporosis',
               'column': 'osteoporosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Osteoporosis', 'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'POA Rheumatoid arthritis',
               'column': 'poa_rheumatoid_arthritis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Rheumatoid arthritis',
                            'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Ankylosing spondylitis',
               'column': 'ankylosing_spondylitis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Ankylosing spondylitis',
                            'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Jaw Movement',
               'column': 'jaw_movement',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Jaw Movement', 'section': 'AIRWAY REVIEW'}]},
              {'label': 'Previous Neck Injury',
               'column': 'previous_neck_injury',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Previous Neck Injury', 'section': 'AIRWAY REVIEW'}]},
              {'label': 'POA Other Endocrine',
               'column': 'poa_other_endocrine_endocrine_review',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Other Endocrine', 'section': 'ENDOCRINE REVIEW'}]},
              {'label': 'Nurse Action Plan',
               'column': 'nurse_action_plan',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Nurse Action Plan', 'section': 'NURSE ACTION PLAN'}]},
              {'label': 'Current occupation',
               'column': 'current_occupation',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'Current occupation', 'section': 'SOCIAL HISTORY'}]},
              {'label': 'POA ADHD',
               'column': 'poa_adhd',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA ADHD', 'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'POA Autism',
               'column': 'poa_autism',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'POA Autism', 'section': 'BEHAVIOURAL/PSYCHOLOGICAL REVIEW'}]},
              {'label': 'Discussed Transport',
               'column': 'discussed_transport',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Discussed Transport',
                            'section': 'INFORMATION DISCUSSED AND PROVIDED'}]},
              {'label': 'History of Falls',
               'column': 'history_of_falls',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'History of Falls', 'section': 'FUNCTIONAL ASSESSMENT'}]},
              {'label': 'Sickle Cell Status',
               'column': 'sickle_cell_status',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Sickle Cell Status', 'section': 'HAEMATOLOGICAL REVIEW'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'hars_hiv_care',
 'display': 'HARS HIV care assessment',
 'source_displays': ['HARS'],
 'elements': [{'label': 'HARS Diagnosis Abroad Year',
               'column': 'hars_diagnosis_abroad_year_baseline',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis Abroad Year', 'section': 'BASELINE'}]},
              {'label': 'HARS CD4 Taken',
               'column': 'hars_cd4_taken_treatment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS CD4 Taken', 'section': 'TREATMENT'}]},
              {'label': 'HIV care date',
               'column': 'hiv_care_date_attendance',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HIV care date', 'section': 'ATTENDANCE'}]},
              {'label': 'HARS Consultation medium used',
               'column': 'hars_consultation_medium_used_attendance',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Consultation medium used', 'section': 'ATTENDANCE'}]},
              {'label': 'HARS Patient care status',
               'column': 'hars_patient_care_status_attendance',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient care status', 'section': 'ATTENDANCE'}]},
              {'label': 'HARS Patient diagnosis indicator (Viraemia)',
               'column': 'hars_patient_diagnosis_indicator_viraemia_attendance',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient diagnosis indicator (Viraemia)',
                            'section': 'ATTENDANCE'}]},
              {'label': 'HIV Care type',
               'column': 'hiv_care_type_attendance',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HIV Care type', 'section': 'ATTENDANCE'}]},
              {'label': 'All professionals involved in care today',
               'column': 'all_professionals_involved_in_care_today',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'All professionals involved in care today',
                            'section': 'ATTENDANCE'}]},
              {'label': 'HARS Diagnosis UK Date',
               'column': 'hars_diagnosis_uk_date_baseline',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis UK Date', 'section': 'BASELINE'}]},
              {'label': 'GP Last Letter',
               'column': 'gp_last_letter',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'GP Last Letter', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS First Seen Date',
               'column': 'hars_first_seen_date_baseline',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First Seen Date', 'section': 'BASELINE'}]},
              {'label': 'HARS Latent TB test performed',
               'column': 'hars_latent_tb_test_performed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Latent TB test performed', 'section': 'BASELINE'}]},
              {'label': 'pt receive PEP last 6 months prior to Diag in UK',
               'column': 'pt_receive_pep_last_6_months_prior_to_diag_in_uk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'pt receive PEP last 6 months prior to Diag in UK',
                            'section': 'BASELINE'}]},
              {'label': 'pt receive PrEP last 6 months prior to Diag in UK',
               'column': 'pt_receive_prep_last_6_months_prior_to_diag_in_uk',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'pt receive PrEP last 6 months prior to Diag in UK',
                            'section': 'BASELINE'}]},
              {'label': 'HARS New diagnosis UK',
               'column': 'hars_new_diagnosis_uk_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS New diagnosis UK', 'section': 'BASELINE'}]},
              {'label': 'HARS Diagnosis setting',
               'column': 'hars_diagnosis_setting_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis setting', 'section': 'BASELINE'}]},
              {'label': 'HARS Patient exposure',
               'column': 'hars_patient_exposure_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient exposure', 'section': 'BASELINE'}]},
              {'label': 'Currently has a social worker',
               'column': 'currently_has_a_social_worker',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Currently has a social worker', 'section': 'VULNERABILITY'}]},
              {'label': 'HARS Prisoner',
               'column': 'hars_prisoner_vulnerability',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Prisoner', 'section': 'VULNERABILITY'}]},
              {'label': 'HARS Sex Worker',
               'column': 'hars_sex_worker_vulnerability',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Sex Worker', 'section': 'VULNERABILITY'}]},
              {'label': 'HARS ARV Band',
               'column': 'hars_arv_band_treatment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS ARV Band', 'section': 'TREATMENT'}]},
              {'label': 'HARS Gender Identity',
               'column': 'hars_gender_identity_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Gender Identity', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HPV Offer Status',
               'column': 'hpv_offer_status',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HPV Offer Status', 'section': 'TREATMENT'}]},
              {'label': 'HARS Gender Identity same at birth',
               'column': 'hars_gender_identity_same_at_birth_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Gender Identity same at birth',
                            'section': 'DEMOGRAPHICS'}]},
              {'label': 'Ethnicity',
               'column': 'ethnicity_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Ethnicity', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS CD4 Taken',
               'column': 'hars_cd4_taken_clinical',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS CD4 Taken', 'section': 'CLINICAL'}]},
              {'label': 'HIV care date',
               'column': 'hiv_care_date_clinical',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HIV care date', 'section': 'CLINICAL'}]},
              {'label': 'HARS Gender Identity',
               'column': 'hars_gender_identity_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Gender Identity', 'section': 'CLINICAL'}]},
              {'label': 'HARS Gender Identity same at birth',
               'column': 'hars_gender_identity_same_at_birth_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Gender Identity same at birth', 'section': 'CLINICAL'}]},
              {'label': 'Ethnicity',
               'column': 'ethnicity_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Ethnicity', 'section': 'CLINICAL'}]},
              {'label': 'HARS New diagnosis UK',
               'column': 'hars_new_diagnosis_uk_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS New diagnosis UK', 'section': 'CLINICAL'}]},
              {'label': 'HARS Patient exposure',
               'column': 'hars_patient_exposure_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient exposure', 'section': 'CLINICAL'}]},
              {'label': 'HARS Diagnosis UK Date',
               'column': 'hars_diagnosis_uk_date_clinical',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis UK Date', 'section': 'CLINICAL'}]},
              {'label': 'HARS Patient care status',
               'column': 'hars_patient_care_status_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient care status', 'section': 'CLINICAL'}]},
              {'label': 'HIV Care type',
               'column': 'hiv_care_type_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HIV Care type', 'section': 'CLINICAL'}]},
              {'label': 'HARS First Seen Date',
               'column': 'hars_first_seen_date_clinical',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First Seen Date', 'section': 'CLINICAL'}]},
              {'label': 'HARS Hepatitis B',
               'column': 'hars_hepatitis_b_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Hepatitis B', 'section': 'CLINICAL'}]},
              {'label': 'HARS Hepatitis C',
               'column': 'hars_hepatitis_c_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Hepatitis C', 'section': 'CLINICAL'}]},
              {'label': 'HARS ARV Band',
               'column': 'hars_arv_band_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS ARV Band', 'section': 'CLINICAL'}]},
              {'label': 'HARS Consultation medium used',
               'column': 'hars_consultation_medium_used_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Consultation medium used', 'section': 'CLINICAL'}]},
              {'label': 'HARS Country of Birth',
               'column': 'hars_country_of_birth_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Country of Birth', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS Disability',
               'column': 'hars_disability_vulnerability',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Disability', 'section': 'VULNERABILITY'}]},
              {'label': 'HARS GP Practice',
               'column': 'hars_gp_practice_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS GP Practice', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS Contact permitted with GP',
               'column': 'hars_contact_permitted_with_gp_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Contact permitted with GP', 'section': 'CLINICAL'}]},
              {'label': 'HARS GP Practice',
               'column': 'hars_gp_practice_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS GP Practice', 'section': 'CLINICAL'}]},
              {'label': 'AIDS defining illness at this visit',
               'column': 'aids_defining_illness_at_this_visit',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AIDS defining illness at this visit',
                            'section': 'DIAGNOSIS'}]},
              {'label': 'Currently on treatment for TB',
               'column': 'currently_on_treatment_for_tb',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Currently on treatment for TB', 'section': 'DIAGNOSIS'}]},
              {'label': 'Currently on treatment from an oncologist',
               'column': 'currently_on_treatment_from_an_oncologist',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Currently on treatment from an oncologist',
                            'section': 'DIAGNOSIS'}]},
              {'label': 'Currently under psychiatric care',
               'column': 'currently_under_psychiatric_care',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Currently under psychiatric care', 'section': 'DIAGNOSIS'}]},
              {'label': 'HARS Hepatitis B',
               'column': 'hars_hepatitis_b_diagnosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Hepatitis B', 'section': 'DIAGNOSIS'}]},
              {'label': 'HARS Hepatitis C',
               'column': 'hars_hepatitis_c_diagnosis',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Hepatitis C', 'section': 'DIAGNOSIS'}]},
              {'label': 'Pregnant or less 30 days since delivery',
               'column': 'pregnant_or_less_30_days_since_delivery',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Pregnant or less 30 days since delivery',
                            'section': 'DIAGNOSIS'}]},
              {'label': 'Severe unstable HIV-related end organ disease',
               'column': 'severe_unstable_hiv_related_end_organ_disease',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Severe unstable HIV-related end organ disease',
                            'section': 'DIAGNOSIS'}]},
              {'label': 'Currently treatment for chronic Hepatitis Bo C',
               'column': 'currently_treatment_for_chronic_hepatitis_bo_c',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Currently treatment for chronic Hepatitis Bo C',
                            'section': 'DIAGNOSIS'}]},
              {'label': 'HARS Country of Birth',
               'column': 'hars_country_of_birth_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Country of Birth', 'section': 'CLINICAL'}]},
              {'label': 'HARS Previous HIV site',
               'column': 'hars_previous_hiv_site_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Previous HIV site', 'section': 'CLINICAL'}]},
              {'label': 'HARS Disability',
               'column': 'hars_disability_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Disability', 'section': 'CLINICAL'}]},
              {'label': 'HARS Previous Negative HIV Test',
               'column': 'hars_previous_negative_hiv_test_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Previous Negative HIV Test', 'section': 'CLINICAL'}]},
              {'label': 'HARS Home Delivery',
               'column': 'hars_home_delivery_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Home Delivery', 'section': 'CLINICAL'}]},
              {'label': 'HARS Seroconversion',
               'column': 'hars_seroconversion_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Seroconversion', 'section': 'CLINICAL'}]},
              {'label': 'HARS Contact permitted with GP',
               'column': 'hars_contact_permitted_with_gp_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Contact permitted with GP', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS Sex Worker',
               'column': 'hars_sex_worker_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Sex Worker', 'section': 'CLINICAL'}]},
              {'label': 'HARS Prisoner',
               'column': 'hars_prisoner_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Prisoner', 'section': 'CLINICAL'}]},
              {'label': 'HARS First ARV UK',
               'column': 'hars_first_arv_uk_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First ARV UK', 'section': 'CLINICAL'}]},
              {'label': 'HARS Previous HIV site',
               'column': 'hars_previous_hiv_site_demographics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Previous HIV site', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS Diagnosis setting',
               'column': 'hars_diagnosis_setting_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis setting', 'section': 'CLINICAL'}]},
              {'label': 'HARS Country of infection',
               'column': 'hars_country_of_infection_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Country of infection', 'section': 'CLINICAL'}]},
              {'label': 'HARS Clinical Trial',
               'column': 'hars_clinical_trial_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Clinical Trial', 'section': 'CLINICAL'}]},
              {'label': 'HARS First ARV UK',
               'column': 'hars_first_arv_uk_treatment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First ARV UK', 'section': 'TREATMENT'}]},
              {'label': 'HARS Patient Clinical Summary',
               'column': 'hars_patient_clinical_summary',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'HARS Patient Clinical Summary', 'section': 'CLINICAL'}]},
              {'label': 'HARS Clinical Trial',
               'column': 'hars_clinical_trial_treatment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Clinical Trial', 'section': 'TREATMENT'}]},
              {'label': 'HARS First ARV start',
               'column': 'hars_first_arv_start_clinical',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First ARV start', 'section': 'CLINICAL'}]},
              {'label': 'HARS Country of infection',
               'column': 'hars_country_of_infection_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Country of infection', 'section': 'BASELINE'}]},
              {'label': 'HARS Site ARV start',
               'column': 'hars_site_arv_start',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Site ARV start', 'section': 'CLINICAL'}]},
              {'label': 'HARS Home Delivery',
               'column': 'hars_home_delivery_treatment',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Home Delivery', 'section': 'TREATMENT'}]},
              {'label': 'HARS Comments/Notes',
               'column': 'hars_comments_notes',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Comments/Notes', 'section': 'CLINICAL'}]},
              {'label': 'HARS Patient diagnosis indicator (Viraemia)',
               'column': 'hars_patient_diagnosis_indicator_viraemia_clinical',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Patient diagnosis indicator (Viraemia)',
                            'section': 'CLINICAL'}]},
              {'label': 'HARS Year UK Arrival',
               'column': 'hars_year_uk_arrival_clinical',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Year UK Arrival', 'section': 'CLINICAL'}]},
              {'label': 'HARS Year UK Arrival',
               'column': 'hars_year_uk_arrival_demographics',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Year UK Arrival', 'section': 'DEMOGRAPHICS'}]},
              {'label': 'HARS Previous Negative HIV Test',
               'column': 'hars_previous_negative_hiv_test_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Previous Negative HIV Test', 'section': 'BASELINE'}]},
              {'label': 'HARS Human papilloma virus (HPV) offer status',
               'column': 'hars_human_papilloma_virus_hpv_offer_status',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Human papilloma virus (HPV) offer status',
                            'section': 'CLINICAL'}]},
              {'label': 'HARS First ARV start',
               'column': 'hars_first_arv_start_treatment',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS First ARV start', 'section': 'TREATMENT'}]},
              {'label': 'HARS AIDS illness',
               'column': 'hars_aids_illness',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'HARS AIDS illness', 'section': 'CLINICAL'}]},
              {'label': 'HARS Last HIV Negative Test Date',
               'column': 'hars_last_hiv_negative_test_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Last HIV Negative Test Date', 'section': 'CLINICAL'}]},
              {'label': 'HARS Diagnosis Abroad Year',
               'column': 'hars_diagnosis_abroad_year_clinical',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Diagnosis Abroad Year', 'section': 'CLINICAL'}]},
              {'label': 'HARS Seroconversion',
               'column': 'hars_seroconversion_baseline',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'HARS Seroconversion', 'section': 'BASELINE'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'diabetes_in_pregnancy',
 'display': 'Diabetes in pregnancy specialist assessment',
 'source_displays': ['Diabetes in Pregnancy Specialist Form'],
 'carry_pregnancy_link': True,
 'elements': [{'label': 'Name of Key Professional (Diabetes)',
               'column': 'name_of_key_professional_diabetes',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Name of Key Professional (Diabetes)',
                            'section': 'US AND ASSESSMENT'}]},
              {'label': 'Date Attended (Diabetes)',
               'column': 'date_attended_diabetes',
               'kind': 'DATE',
               'cardinality': 'multi',
               'matches': [{'label': 'Date Attended (Diabetes)', 'section': 'US AND ASSESSMENT'}]},
              {'label': 'Advice and Future Insulin Doses',
               'column': 'advice_and_future_insulin_doses',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Advice and Future Insulin Doses',
                            'section': 'US AND ASSESSMENT'}]},
              {'label': 'Current Insulin Dose',
               'column': 'current_insulin_dose',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Current Insulin Dose', 'section': 'US AND ASSESSMENT'}]},
              {'label': 'Gestation at Attendance (Diabetes)',
               'column': 'gestation_at_attendance_diabetes',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Gestation at Attendance (Diabetes)',
                            'section': 'US AND ASSESSMENT'}]},
              {'label': 'Home Blood Test Results',
               'column': 'home_blood_test_results',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Home Blood Test Results', 'section': 'US AND ASSESSMENT'}]},
              {'label': 'Type of Diabetes',
               'column': 'type_of_diabetes',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Type of Diabetes', 'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Method Of Diagnosis (Diabetes)',
               'column': 'method_of_diagnosis_diabetes',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'Method Of Diagnosis (Diabetes)',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Fasting Result From OGTT',
               'column': 'fasting_result_from_ogtt',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'Fasting Result From OGTT',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'OGTT Result After 2 Hours',
               'column': 'ogtt_result_after_2_hours',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'OGTT Result After 2 Hours',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Date of 1st OGTT',
               'column': 'date_of_1st_ogtt',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'Date of 1st OGTT', 'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Current Medication',
               'column': 'current_medication',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Current Medication', 'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Prescription Drugs Route',
               'column': 'prescription_drugs_route',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Prescription Drugs Route',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': '1st Antenatal Diabetic Appointment',
               'column': 'item_1st_antenatal_diabetic_appointment',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': '1st Antenatal Diabetic Appointment',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Prescription Drugs Dose',
               'column': 'prescription_drugs_dose',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Prescription Drugs Dose',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Prescription Drugs Frequency',
               'column': 'prescription_drugs_frequency',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'Prescription Drugs Frequency',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Prescription Drugs Comment',
               'column': 'prescription_drugs_comment',
               'kind': 'TEXT',
               'cardinality': 'multi',
               'matches': [{'label': 'Prescription Drugs Comment',
                            'section': 'DIABETES IN PREGNANCY'}]},
              {'label': 'Random Blood Sugar Date',
               'column': 'random_blood_sugar_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'Random Blood Sugar Date',
                            'section': 'DIABETES IN PREGNANCY'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'acute_haemodialysis_treatment',
 'display': 'Acute haemodialysis treatment record',
 'source_displays': ['Acute Haemodialysis Treatment Record'],
 'elements': [{'label': 'AHTR_Prescribed fluid removal (L)',
               'column': 'ahtr_prescribed_fluid_removal_l',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Prescribed fluid removal (L)',
                            'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Dialysis date',
               'column': 'ahtr_dialysis_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis date', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Potassium',
               'column': 'ahtr_potassium',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Potassium', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Bicarbonate',
               'column': 'ahtr_bicarbonate',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Bicarbonate', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Anticoagulation',
               'column': 'ahtr_anticoagulation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Anticoagulation', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Dialysate temperature',
               'column': 'ahtr_dialysate_temperature',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysate temperature', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Duration of dialysis (hours)',
               'column': 'ahtr_duration_of_dialysis_hours',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Duration of dialysis (hours)',
                            'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Calcium',
               'column': 'ahtr_calcium',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Calcium', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Sodium conductivity',
               'column': 'ahtr_sodium_conductivity',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Sodium conductivity', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Additional instructions',
               'column': 'ahtr_additional_instructions_prescription',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Additional instructions', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Dialysis machine number',
               'column': 'ahtr_dialysis_machine_number',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis machine number',
                            'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dialysis start time',
               'column': 'ahtr_dialysis_start_time',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis start time', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dialysis hours',
               'column': 'ahtr_dialysis_hours',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis hours', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Isolate machine',
               'column': 'ahtr_isolate_machine',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Isolate machine', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Pulse / HR_Pre Dialysis',
               'column': 'ahtr_pulse_hr_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Pulse / HR_Pre Dialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Blood pressureSP(lying)_Pre dialysis',
               'column': 'ahtr_blood_pressuresp_lying_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood pressureSP(lying)_Pre dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Blood pressureDP (lying)_Pre dialysis',
               'column': 'ahtr_blood_pressuredp_lying_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood pressureDP (lying)_Pre dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Type',
               'column': 'ahtr_type',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Type', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dialysis end time',
               'column': 'ahtr_dialysis_end_time',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis end time', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Temperature_Pre Dialysis',
               'column': 'ahtr_temperature_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Temperature_Pre Dialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Dialysate flow',
               'column': 'ahtr_dialysate_flow',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysate flow', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dialysate solution',
               'column': 'ahtr_dialysate_solution',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysate solution', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Pulse / HR_Post Dialysis',
               'column': 'ahtr_pulse_hr_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Pulse / HR_Post Dialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Blood pressureSP (lying)_Post dialysis',
               'column': 'ahtr_blood_pressuresp_lying_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood pressureSP (lying)_Post dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Dialysis commenced by',
               'column': 'ahtr_dialysis_commenced_by',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis commenced by', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Blood pressureDP (lying)_Post dialysis',
               'column': 'ahtr_blood_pressuredp_lying_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood pressureDP (lying)_Post dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Dialyser_DD',
               'column': 'ahtr_dialyser_dd',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialyser_DD', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Temperature_Post Dialysis',
               'column': 'ahtr_temperature_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Temperature_Post Dialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Dialysis treatment record documented by',
               'column': 'ahtr_dialysis_treatment_record_documented_by',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis treatment record documented by',
                            'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Recent Travel',
               'column': 'ahtr_recent_travel',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Recent Travel', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Litres processed',
               'column': 'ahtr_litres_processed',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Litres processed', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Blood transfusion',
               'column': 'ahtr_blood_transfusion',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood transfusion', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Citralock Dose',
               'column': 'ahtr_citralock_dose',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Citralock Dose', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_IV antibiotics',
               'column': 'ahtr_iv_antibiotics',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_IV antibiotics', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTRSJ_Additional Comments',
               'column': 'ahtrsj_additional_comments',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTRSJ_Additional Comments', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dialyser',
               'column': 'ahtr_dialyser',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialyser', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Other anticoagulation',
               'column': 'ahtr_other_anticoagulation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Other anticoagulation', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Home dialysis unit',
               'column': 'ahtr_home_dialysis_unit',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Home dialysis unit', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Inhixa Dose',
               'column': 'ahtr_inhixa_dose',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Inhixa Dose', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Citralock volume a',
               'column': 'ahtr_citralock_volume_a',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Citralock volume a', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Citralock volume b',
               'column': 'ahtr_citralock_volume_b',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Citralock volume b', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Blood Results',
               'column': 'ahtr_blood_results',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Blood Results',
                            'section': 'TREATMENT RECORD & BLOOD RESULTS'}]},
              {'label': 'AHTR_Reason why weight not recorded(PD)',
               'column': 'ahtr_reason_why_weight_not_recorded_pd',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Reason why weight not recorded(PD)',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Reason why weight not recorded(POD)',
               'column': 'ahtr_reason_why_weight_not_recorded_pod',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Reason why weight not recorded(POD)',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Size of needles',
               'column': 'ahtr_size_of_needles',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Size of needles', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Respiration rate_Pre dialysis',
               'column': 'ahtr_respiration_rate_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Respiration rate_Pre dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Drug given',
               'column': 'ahtr_drug_given',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Drug given', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Weight (Pre Dialysis)',
               'column': 'ahtr_weight_pre_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Weight (Pre Dialysis)', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Dialysis session date',
               'column': 'ahtr_dialysis_session_date',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dialysis session date',
                            'section': 'TREATMENT RECORD & BLOOD RESULTS'}]},
              {'label': 'AHTR_Respiration rate_Post dialysis',
               'column': 'ahtr_respiration_rate_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Respiration rate_Post dialysis',
                            'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Additional instructions',
               'column': 'ahtr_additional_instructions_dialysis_details',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Additional instructions',
                            'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Weight (Post Dialysis)',
               'column': 'ahtr_weight_post_dialysis',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Weight (Post Dialysis)', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Other Anticoagulation Other',
               'column': 'ahtr_other_anticoagulation_other',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Other Anticoagulation Other',
                            'section': 'DIALYSIS DETAILS'}]},
              {'label': 'BM_PreDialysis',
               'column': 'bm_predialysis',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'BM_PreDialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Number of units',
               'column': 'ahtr_number_of_units',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Number of units', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Dry weight (kg)',
               'column': 'ahtr_dry_weight_kg',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Dry weight (kg)', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Total Fluid Loss',
               'column': 'ahtr_total_fluid_loss',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Total Fluid Loss', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Estimated dry weight (kg)',
               'column': 'ahtr_estimated_dry_weight_kg',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Estimated dry weight (kg)', 'section': 'PRESCRIPTION'}]},
              {'label': 'BM_PostDialysis',
               'column': 'bm_postdialysis',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'BM_PostDialysis', 'section': 'OBSERVATIONS'}]},
              {'label': 'AHTR_Anticoagulation Other',
               'column': 'ahtr_anticoagulation_other',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Anticoagulation Other', 'section': 'PRESCRIPTION'}]},
              {'label': 'AHTR_Citralock dose Other',
               'column': 'ahtr_citralock_dose_other',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Citralock dose Other', 'section': 'DIALYSIS DETAILS'}]},
              {'label': 'AHTR_Tinzaparin dose (iu)',
               'column': 'ahtr_tinzaparin_dose_iu',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AHTR_Tinzaparin dose (iu)', 'section': 'DIALYSIS DETAILS'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'sbar_escalation',
 'display': 'SBAR escalation and response',
 'source_displays': ['SBAR Escalation Form', 'SBAR Escalation Response Form'],
 'elements': [{'label': 'SBAR Escalation Required',
               'column': 'sbar_escalation_required',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Escalation Required', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Identify yourself',
               'column': 'sbar_identify_yourself',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Identify yourself', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SEP Time of deterioration',
               'column': 'sep_time_of_deterioration',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'SEP Time of deterioration', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Agreed plan',
               'column': 'sbar_agreed_plan',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Agreed plan', 'section': 'SBAR ESCALATION'},
                           {'label': 'SBAR Agreed plan', 'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Outline concern',
               'column': 'sbar_outline_concern',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Outline concern', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Recommendation',
               'column': 'sbar_recommendation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Recommendation', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Role',
               'column': 'sbar_role',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Role', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Other clinical signs',
               'column': 'sbar_other_clinical_signs',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Other clinical signs', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Agree timeframe for review by team/CCOT/ART',
               'column': 'sbar_agree_timeframe_for_review_by_team_ccot_art',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Agree timeframe for review by team/CCOT/ART',
                            'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Ask what interventions you can do?',
               'column': 'sbar_ask_what_interventions_you_can_do',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Ask what interventions you can do?',
                            'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Escalation Role',
               'column': 'sbar_escalation_role',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Escalation Role',
                            'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Referral reason',
               'column': 'sbar_referral_reason',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Referral reason',
                            'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Advice given',
               'column': 'sbar_advice_given',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Advice given', 'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Contact Type',
               'column': 'sbar_contact_type',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Contact Type', 'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Escalation response',
               'column': 'sbar_escalation_response',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'SBAR Escalation response',
                            'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR State your clinical impression',
               'column': 'sbar_state_your_clinical_impression',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR State your clinical impression',
                            'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Name of person spoken to',
               'column': 'sbar_name_of_person_spoken_to',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Name of person spoken to',
                            'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Additional information',
               'column': 'sbar_additional_information',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Additional information', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Agreed timeframe',
               'column': 'sbar_agreed_timeframe',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Agreed timeframe',
                            'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR What else do you need?',
               'column': 'sbar_what_else_do_you_need',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR What else do you need?', 'section': 'SBAR ESCALATION'}]},
              {'label': 'SBAR Referrer',
               'column': 'sbar_referrer',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Referrer', 'section': 'SBAR ESCALATION RESPONSE'}]},
              {'label': 'SBAR Rationale for not escalating',
               'column': 'sbar_rationale_for_not_escalating',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'SBAR Rationale for not escalating',
                            'section': 'SBAR ESCALATION'}]}]}
)

In [0]:
FORM_CONFIGS.append(
{'slug': 'major_trauma_rehabilitation',
 'display': 'Adult major-trauma rehabilitation data',
 'source_displays': ['Adult Major Trauma Rehabilitation Data'],
 'elements': [{'label': 'AMTD Does the patient have rehabilitation needs?',
               'column': 'amtd_does_the_patient_have_rehabilitation_needs',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Does the patient have rehabilitation needs?',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Phsyical impairment requiring rehab',
               'column': 'amtd_phsyical_impairment_requiring_rehab',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AMTD Phsyical impairment requiring rehab',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Employment (pre-accident)',
               'column': 'amtd_employment_pre_accident',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Employment (pre-accident)',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Date Completed',
               'column': 'mtd_date_completed',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Date Completed',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'AMTD Rehabilitation Prescription been developed',
               'column': 'amtd_rehabilitation_prescription_been_developed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Rehabilitation Prescription been developed',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Rehab Prescription discuss Patient',
               'column': 'amtd_rehab_prescription_discuss_patient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Rehab Prescription discuss Patient',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Rehab Prescription given to GP',
               'column': 'amtd_rehab_prescription_given_to_gp',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AMTD Rehab Prescription given to GP',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Rehab Prescription given next care provider',
               'column': 'amtd_rehab_prescription_given_next_care_provider',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Rehab Prescription given next care provider',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Summary of presentation',
               'column': 'mtd_summary_of_presentation',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Summary of presentation',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': "AMTD the patient's rehab need assessed",
               'column': 'amtd_the_patient_s_rehab_need_assessed',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': "AMTD the patient's rehab need assessed",
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD transferred to appropriate facility',
               'column': 'amtd_transferred_to_appropriate_facility',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD transferred to appropriate facility',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Rehab Prescription Summary of plan',
               'column': 'mtd_rehab_prescription_summary_of_plan',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Rehab Prescription Summary of plan',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Hospital Site',
               'column': 'mtd_hospital_site',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Hospital Site',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Name of Ward',
               'column': 'mtd_name_of_ward',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Name of Ward',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'AMTD Onwards referrals made',
               'column': 'amtd_onwards_referrals_made',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Onwards referrals made',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD SiteTelephone Number RLH',
               'column': 'mtd_sitetelephone_number_rlh',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'MTD SiteTelephone Number RLH',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Your Consultant',
               'column': 'mtd_your_consultant',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Your Consultant',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Ext No',
               'column': 'mtd_ext_no',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Ext No',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'AMTD Patients actual discharge destination',
               'column': 'amtd_patients_actual_discharge_destination',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Patients actual discharge destination',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Community Rehabilitation',
               'column': 'amtd_community_rehabilitation',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Community Rehabilitation',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD the patients destination',
               'column': 'amtd_the_patients_destination',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD the patients destination',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Contact info for your local Rehab services',
               'column': 'mtd_contact_info_for_your_local_rehab_services',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Contact info for your local Rehab services',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'AMTD Community Rehabilitation need',
               'column': 'amtd_community_rehabilitation_need',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Community Rehabilitation need',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Date transferred discharged',
               'column': 'mtd_date_transferred_discharged',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Date transferred discharged',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Waiting List',
               'column': 'mtd_waiting_list',
               'kind': 'NUMERIC',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Waiting List',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'AMTD Cognitive mood disturb req rehab',
               'column': 'amtd_cognitive_mood_disturb_req_rehab',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AMTD Cognitive mood disturb req rehab',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Referral sent on',
               'column': 'mtd_referral_sent_on',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Referral sent on',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Date referral sent for further rehab',
               'column': 'mtd_date_referral_sent_for_further_rehab',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Date referral sent for further rehab',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Specialist outpatient',
               'column': 'amtd_specialist_outpatient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Specialist outpatient',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Specialist outpatient need',
               'column': 'amtd_specialist_outpatient_need',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Specialist outpatient need',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Psychosocial issues',
               'column': 'amtd_psychosocial_issues',
               'kind': 'CODED',
               'cardinality': 'multi',
               'matches': [{'label': 'AMTD Psychosocial issues',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Specialist inpatient',
               'column': 'amtd_specialist_inpatient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Specialist inpatient',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD Advice and Education',
               'column': 'mtd_advice_and_education',
               'kind': 'TEXT',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Advice and Education',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]},
              {'label': 'MTD Date of review by rehab unit',
               'column': 'mtd_date_of_review_by_rehab_unit',
               'kind': 'DATE',
               'cardinality': 'single',
               'matches': [{'label': 'MTD Date of review by rehab unit',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'PMTD What is the reason for variance',
               'column': 'pmtd_what_is_the_reason_for_variance',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'PMTD What is the reason for variance',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Non-specialist inpatient',
               'column': 'amtd_non_specialist_inpatient',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Non-specialist inpatient',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Specialist inpatient need',
               'column': 'amtd_specialist_inpatient_need',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Specialist inpatient need',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'AMTD Non-specialist inpatient need',
               'column': 'amtd_non_specialist_inpatient_need',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'AMTD Non-specialist inpatient need',
                            'section': 'ADULT MAJOR TRAUMA REHABILITATION DATA'}]},
              {'label': 'MTD SiteTelephone Number WX',
               'column': 'mtd_sitetelephone_number_wx',
               'kind': 'CODED',
               'cardinality': 'single',
               'matches': [{'label': 'MTD SiteTelephone Number WX',
                            'section': 'PATIENT COPY REHABILITATION PRESCRIPTION'}]}]}
)


In [0]:
def _norm(column):
    return F.upper(F.trim(column))


In [0]:
def _qident(value):
    return "`" + value.replace("`", "``") + "`"


In [0]:
def _qtable(name):
    return ".".join(_qident(value) for value in name.split("."))


In [0]:
def _value_expr(kind):
    if kind == "NUMERIC":
        return F.col("r.response_value_number")
    if kind == "DATE":
        return F.col("r.response_value_datetime")
    return F.coalesce(
        F.col("r.response_coding_display"),
        F.col("r.response_value_text"),
    )


In [0]:
def _element_matches(element):
    if element.get("matches"):
        return element["matches"]
    labels = [element["label"]] + element.get("aliases", [])
    return [{"label": label, "section": element.get("section")} for label in labels]


In [0]:
def _validate_config(cfg):
    columns = [element["column"] for element in cfg["elements"]]
    assert len(columns) == len(set(columns)), f"{cfg['slug']}: duplicate output columns"
    assert all(element["kind"] in {"NUMERIC", "CODED", "TEXT", "DATE"} for element in cfg["elements"])
    assert all(element["cardinality"] in {"single", "multi"} for element in cfg["elements"])

    keys = []
    for element in cfg["elements"]:
        for match in _element_matches(element):
            key = (
                match["label"].strip().upper(),
                (match.get("section") or "").strip().upper() or None,
            )
            keys.append(key)
    assert len(keys) == len(set(keys)), f"{cfg['slug']}: duplicate label/section match keys"


In [0]:
def build_form_table(cfg):
    """Build to a staging table; the driver validates before replacing target."""
    _validate_config(cfg)
    tgt = f"{TARGET_SCHEMA}.clinical_form_{cfg['slug']}"
    stg = f"{STAGING_SCHEMA}.clinical_form_{cfg['slug']}__stg"

    rows = []
    for element in cfg["elements"]:
        for match in _element_matches(element):
            rows.append(
                (
                    match["label"].strip().upper(),
                    (match.get("section") or "").strip().upper() or None,
                    element["column"],
                    element["kind"],
                    element["cardinality"],
                )
            )
    map_df = spark.createDataFrame(
        rows,
        "label_norm string, section_norm string, column string, kind string, cardinality string",
    )

    base = spark.table(SOURCE_TABLE).where(
        F.col("source_display").isin(cfg["source_displays"])
    )

    exploded = base.select(
        "patient_event_id",
        "source_display",
        F.explode(
            F.from_json(F.to_json(F.col("responses")), RESPONSE_SCHEMA)
        ).alias("r"),
    )

    prov = (
        spark.table(MAP_LANE)
        .where(F.col("FORM_DESC_TXT").isin(cfg["source_displays"]))
        .select(
            F.col("DOC_RESPONSE_KEY").alias("_prov_key"),
            *[
                F.col(name).alias(f"_prov_{name.lower()}")
                for name in PROVENANCE_COLS
            ],
        )
    )
    enriched = exploded.join(
        prov,
        F.col("r.response_id") == F.col("_prov_key"),
        "left",
    ).drop("_prov_key")

    matched = enriched.join(
        F.broadcast(map_df),
        (_norm(F.col("r.element_label")) == F.col("label_norm"))
        & (
            F.col("section_norm").isNull()
            | (_norm(F.col("r.section")) == F.col("section_norm"))
        ),
        "left",
    )

    ambiguous_wildcards = (
        matched.where(
            F.col("column").isNotNull() & F.col("section_norm").isNull()
        )
        .groupBy("source_display", "column")
        .agg(F.collect_set(_norm(F.col("r.section"))).alias("sections"))
        .where(F.size("sections") > 1)
        .collect()
    )
    assert not ambiguous_wildcards, (
        f"{tgt}: wildcard labels span multiple sections: "
        f"{[(row['source_display'], row['column'], row['sections']) for row in ambiguous_wildcards]}"
    )

    has_value = (
        F.col("r.response_value_number").isNotNull()
        | F.col("r.response_value_text").isNotNull()
        | F.col("r.response_value_datetime").isNotNull()
        | F.col("_prov_value_concept_id").isNotNull()
    )
    window = (
        Window.partitionBy("patient_event_id", "column")
        .orderBy(
            F.col("r.active_ind").desc_nulls_last(),
            has_value.desc(),
            F.col("r.sequence").desc_nulls_last(),
            F.col("r.response_id").desc_nulls_last(),
        )
    )
    ranked = matched.withColumn("_rn", F.row_number().over(window))

    picked = (
        F.when(F.col("column").isNull(), F.lit(False))
        .when(F.col("cardinality") == "single", F.col("_rn") == 1)
        .otherwise(F.coalesce(F.col("r.active_ind"), F.lit(False)))
    )
    ranked = ranked.withColumn("_picked", picked)

    provenance_json = F.to_json(
        F.struct(
            F.col("_prov_value_mapping_source").alias("source"),
            F.col("_prov_powerform_mapping_rule_id").alias("rule_id"),
            F.col("_prov_powerform_mapping_version").alias("ruleset_version"),
            F.col("_prov_canonical_value_mapping_rule_id").alias("canonical_rule_id"),
            F.col("_prov_canonical_match_status").alias("match_status"),
            F.col("_prov_question_concept_id").alias("question_concept_id"),
            F.col("_prov_unit_concept_id").alias("unit_concept_id"),
        )
    )

    aggregations = []
    comments = {}
    multi_columns = []
    for element in cfg["elements"]:
        column = element["column"]
        kind = element["kind"]
        cardinality = element["cardinality"]
        take = (F.col("column") == column) & F.col("_picked")
        if cardinality == "single":
            aggregations.extend(
                [
                    F.max(F.when(take, _value_expr(kind))).alias(column),
                    F.max(
                        F.when(take, F.col("r.response_value_text"))
                    ).alias(f"{column}_text"),
                    F.max(
                        F.when(take, F.col("_prov_value_concept_id"))
                    ).alias(f"{column}_concept_id"),
                    F.max(F.when(take, provenance_json)).alias(
                        f"{column}_provenance"
                    ),
                ]
            )
        else:
            item = F.when(
                take,
                F.struct(
                    F.col("r.sequence").alias("sequence"),
                    F.col("r.response_id").alias("response_id"),
                    _value_expr(kind).cast("string").alias("value"),
                    F.col("r.response_value_text").alias("text"),
                    F.col("_prov_value_concept_id").alias("concept_id"),
                    F.col("_prov_value_mapping_source").alias("mapping_source"),
                    F.col("_prov_powerform_mapping_rule_id").alias(
                        "mapping_rule_id"
                    ),
                ),
            )
            aggregations.append(
                F.array_sort(F.collect_list(item)).alias(f"_{column}_arr")
            )
            multi_columns.append(column)

        match_text = "; ".join(
            f"{match['label']} [{match.get('section') or 'ANY'}]"
            for match in _element_matches(element)
        )
        comments[column] = (
            f"{element['label']} — {kind}/{cardinality}; matches: {match_text}"
        )

    if cfg.get("carry_pregnancy_link"):
        aggregations.extend(
            [
                F.max(F.col("r.pregnancy_id")).alias("pregnancy_id"),
                F.max(F.col("r.pregnancy_match_method")).alias(
                    "pregnancy_match_method"
                ),
                F.countDistinct(
                    F.when(
                        F.col("r.pregnancy_id").isNotNull(),
                        F.col("r.pregnancy_id"),
                    )
                ).alias("_pregnancy_id_variants"),
            ]
        )
        comments["pregnancy_id"] = (
            "Pregnancy episode linked on the source form responses."
        )
        comments["pregnancy_match_method"] = (
            "Method used to link the source form response to pregnancy."
        )

    aggregations.extend(
        [
            F.to_json(
                F.array_sort(
                    F.collect_list(F.when(~F.col("_picked"), F.col("r")))
                )
            ).alias("other_responses"),
            F.sum(F.col("_picked").cast("long")).alias(
                "_consumed_response_rows"
            ),
            F.sum((~F.col("_picked")).cast("long")).alias(
                "_residual_response_rows"
            ),
        ]
    )

    aggregate = ranked.groupBy("patient_event_id").agg(*aggregations)
    for column in multi_columns:
        aggregate = (
            aggregate.withColumn(column, F.to_json(F.col(f"_{column}_arr")))
            .withColumn(
                f"{column}_text",
                F.array_join(
                    F.transform(
                        F.col(f"_{column}_arr"),
                        lambda item: item["value"],
                    ),
                    "|",
                ),
            )
            .drop(f"_{column}_arr")
        )

    output = (
        base.select(*HOUSE_COLUMNS)
        .join(aggregate, "patient_event_id", "left")
        .withColumn(
            "_consumed_response_rows",
            F.coalesce(F.col("_consumed_response_rows"), F.lit(0)),
        )
        .withColumn(
            "_residual_response_rows",
            F.coalesce(F.col("_residual_response_rows"), F.lit(0)),
        )
        .withColumn(
            "other_responses",
            F.coalesce(F.col("other_responses"), F.lit("[]")),
        )
    )
    (
        output.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(stg)
    )
    return stg, tgt, comments


In [0]:
def validate_form_table(cfg, stg):
    source_count = (
        spark.table(SOURCE_TABLE)
        .where(F.col("source_display").isin(cfg["source_displays"]))
        .count()
    )
    table = spark.table(stg)
    row_count = table.count()
    distinct_pk_count = table.select("patient_event_id").distinct().count()
    null_pk_count = table.where(F.col("patient_event_id").isNull()).count()
    bad_reconciliation = table.where(
        (
            F.col("_consumed_response_rows")
            + F.col("_residual_response_rows")
        )
        != F.coalesce(F.col("response_row_count"), F.lit(0))
    ).count()
    consumed_response_rows = table.agg(
        F.coalesce(F.sum("_consumed_response_rows"), F.lit(0)).alias("n")
    ).first()["n"]
    bad_pregnancy_link = (
        table.where(F.col("_pregnancy_id_variants") > 1).count()
        if cfg.get("carry_pregnancy_link")
        else 0
    )

    checks = [
        ("row_count_matches_source", row_count == source_count, f"{row_count} vs {source_count}"),
        ("pk_unique", row_count == distinct_pk_count, f"{row_count} vs {distinct_pk_count}"),
        ("pk_not_null", null_pk_count == 0, str(null_pk_count)),
        ("has_rows", row_count > 0, str(row_count)),
        (
            "has_named_responses",
            consumed_response_rows > 0,
            str(consumed_response_rows),
        ),
        (
            "per_instance_response_conservation",
            bad_reconciliation == 0,
            f"{bad_reconciliation} instances where consumed+residual != response_row_count",
        ),
    ]
    if cfg.get("carry_pregnancy_link"):
        checks.append(
            (
                "pregnancy_link_consistent",
                bad_pregnancy_link == 0,
                f"{bad_pregnancy_link} instances with multiple pregnancy ids",
            )
        )

    assert all(passed for _, passed, _ in checks), (
        f"validation failed for {stg}: {checks}"
    )

In [0]:
def require_columns(table_name, required):
    available = set(spark.table(table_name).columns)
    missing = sorted(set(required) - available)
    if missing:
        raise RuntimeError(f"{table_name} is missing required columns: {missing}")


def _publish_form_instrument(config):
    staging_table, target_table, column_comments = build_form_table(config)
    validate_form_table(config, staging_table)

    spark.sql(
        f"CREATE OR REPLACE TABLE {_qtable(target_table)} AS "
        f"SELECT * FROM {_qtable(staging_table)}"
    )
    spark.sql(f"DROP TABLE {_qtable(staging_table)}")
    spark.sql(
        f"ALTER TABLE {_qtable(target_table)} SET TBLPROPERTIES ("
        "delta.enableRowTracking=true, "
        "delta.enableChangeDataFeed=true, "
        "delta.enableDeletionVectors=true)"
    )

    safe_display = config["display"].replace("'", "''")
    spark.sql(
        f"COMMENT ON TABLE {_qtable(target_table)} IS '{safe_display} — one row per form "
        "instance (patient_event_id), all source record statuses retained. Generated "
        "from clinical_form and map_powerform_assessment_item; mapping provenance "
        "is carried beside named values.'"
    )
    for column, comment in column_comments.items():
        safe_comment = comment.replace("'", "''")
        spark.sql(
            f"ALTER TABLE {_qtable(target_table)} ALTER COLUMN {_qident(column)} "
            f"COMMENT '{safe_comment}'"
        )
    return target_table


In [0]:
def build_form_instruments():
    require_columns(
        SOURCE_TABLE,
        {
            "patient_event_id",
            "fact_row_id",
            "person_id",
            "event_datetime",
            "source_display",
            "responses",
            "response_row_count",
            "record_status",
        },
    )
    require_columns(
        MAP_LANE,
        {
            "DOC_RESPONSE_KEY",
            "FORM_DESC_TXT",
            "QUESTION_CONCEPT_ID",
            "VALUE_CONCEPT_ID",
            "UNIT_CONCEPT_ID",
            "POWERFORM_MAPPING_RULE_ID",
        },
    )

    selected = FORM_CONFIGS
    if not selected:
        raise RuntimeError("no form instruments selected")

    built = []
    for config in selected:
        target = _publish_form_instrument(config)
        built.append(target)
        print("built", target)

    return {"count": len(built), "tables": built}


In [0]:
def ensure_patient_event_clustering():
    catalog = _qident("4_prod")
    candidates = spark.sql(f"""
        SELECT table_name
        FROM {catalog}.information_schema.tables
        WHERE table_schema = 'silver'
          AND table_name RLIKE
              '^__materialization_mat_[0-9a-f_]+_events_patient_event_[0-9]+$'
        ORDER BY table_name
    """).collect()

    if len(candidates) != 1:
        names = [row.table_name for row in candidates]
        raise RuntimeError(
            "expected exactly one physical events_patient_event materialization; "
            f"found {names}"
        )

    physical = f"4_prod.silver.{candidates[0].table_name}"
    detail = spark.sql(f"DESCRIBE DETAIL {_qtable(physical)}").first()
    current = [str(value).lower() for value in (detail.clusteringColumns or [])]
    expected = ["person_id", "event_datetime"]

    if current != expected:
        spark.sql(
            f"ALTER TABLE {_qtable(physical)} SET TBLPROPERTIES "
            "('delta.dataSkippingStatsColumns'='person_id,event_datetime')"
        )
        spark.sql(f"ANALYZE TABLE {_qtable(physical)} COMPUTE DELTA STATISTICS")
        spark.sql(
            f"ALTER TABLE {_qtable(physical)} "
            "CLUSTER BY (person_id, event_datetime)"
        )
        detail = spark.sql(f"DESCRIBE DETAIL {_qtable(physical)}").first()
        current = [
            str(value).lower() for value in (detail.clusteringColumns or [])
        ]

    if current != expected:
        raise RuntimeError(
            f"patient-event clustering verification failed for {physical}: {current}"
        )

    return {"table": physical, "columns": current}

In [0]:
result = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "form_instruments": build_form_instruments(),
    "patient_event_clustering": ensure_patient_event_clustering(),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

print(json.dumps(result, indent=2, sort_keys=True))